In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folders = [
    f"{BASE}/data/raw/manual/wikisource",
    f"{BASE}/data/processed/manual/wikisource",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print("CREATED:", folder)

CREATED: /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/wikisource
CREATED: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/wikisource


In [ ]:
DUMP_URL = (
    "https://dumps.wikimedia.org/"
    "tewikisource/20260801/"
    "tewikisource-20260801-pages-articles.xml.bz2"
)

DUMP_FILE = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/"
    "tewikisource-20260801-pages-articles.xml.bz2"
)

print("URL:", DUMP_URL)
print("Destination:", DUMP_FILE)

URL: https://dumps.wikimedia.org/tewikisource/20260801/tewikisource-20260801-pages-articles.xml.bz2
Destination: /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/wikisource/tewikisource-20260801-pages-articles.xml.bz2


In [ ]:
import requests

r = requests.head(
    DUMP_URL,
    allow_redirects=True,
    timeout=30
)

print("Status:", r.status_code)
print("Content-Length:", r.headers.get("Content-Length"))
print("Content-Type:", r.headers.get("Content-Type"))

Status: 403
Content-Length: 135
Content-Type: application/octet-stream


In [ ]:
import os

DUMP_URL = (
    "https://dumps.wikimedia.org/"
    "tewikisource/20260801/"
    "tewikisource-20260801-pages-articles.xml.bz2"
)

DUMP_FILE = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/"
    "tewikisource-20260801-pages-articles.xml.bz2"
)

!wget \
  --user-agent="Mozilla/5.0" \
  --continue \
  -O "{DUMP_FILE}" \
  "{DUMP_URL}"

--2026-08-15 09:11:10--  https://dumps.wikimedia.org/tewikisource/20260801/tewikisource-20260801-pages-articles.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118868481 (113M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/wikisource/tewikisource-20260801-pages-articles.xml.bz2’

/content/drive/MyDr 100%[===================>] 113.36M  4.54MB/s    in 27s     

2026-08-15 09:11:37 (4.26 MB/s) - ‘/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/wikisource/tewikisource-20260801-pages-articles.xml.bz2’ saved [118868481/118868481]



In [ ]:
import os

print("Exists:", os.path.exists(DUMP_FILE))

if os.path.exists(DUMP_FILE):
    print(
        "Size:",
        round(os.path.getsize(DUMP_FILE) / (1024**3), 3),
        "GB"
    )

Exists: True
Size: 0.111 GB


In [ ]:
import bz2
import xml.etree.ElementTree as ET
import os
import time

print("Dump exists:", os.path.exists(DUMP_FILE))
print(
    "Size:",
    round(os.path.getsize(DUMP_FILE) / (1024**2), 2),
    "MB"
)

pages = 0
namespaces = {}
redirects = 0
with_text = 0

start = time.time()

with bz2.open(DUMP_FILE, "rb") as f:

    for event, elem in ET.iterparse(
        f,
        events=("end",)
    ):

        if not elem.tag.endswith("page"):
            continue

        pages += 1

        ns = None
        text = ""

        for child in elem.iter():

            if child.tag.endswith("ns"):
                ns = child.text or ""

            elif child.tag.endswith("text"):
                text = child.text or ""

        namespaces[ns] = namespaces.get(ns, 0) + 1

        if text.strip():
            with_text += 1

        if text.lstrip().lower().startswith("#redirect"):
            redirects += 1

        if pages % 10_000 == 0:
            elapsed = time.time() - start
            print(
                f"Pages inspected: {pages:,} | "
                f"Rate: {pages/elapsed:.1f} pages/sec"
            )

        elem.clear()

print()
print("=" * 70)
print("WIKISOURCE DUMP INSPECTION")
print("=" * 70)

print("Total pages:", pages)
print("Pages with text:", with_text)
print("Redirects:", redirects)

print("\nNamespaces:")
for ns, count in sorted(
    namespaces.items(),
    key=lambda x: int(x[0]) if x[0].isdigit() else 999
):
    print(f"  {ns}: {count:,}")

Dump exists: True
Size: 113.36 MB
Pages inspected: 10,000 | Rate: 1167.9 pages/sec
Pages inspected: 20,000 | Rate: 1276.4 pages/sec
Pages inspected: 30,000 | Rate: 1551.4 pages/sec
Pages inspected: 40,000 | Rate: 1661.1 pages/sec
Pages inspected: 50,000 | Rate: 1757.0 pages/sec
Pages inspected: 60,000 | Rate: 1835.7 pages/sec
Pages inspected: 70,000 | Rate: 1857.4 pages/sec
Pages inspected: 80,000 | Rate: 1920.9 pages/sec
Pages inspected: 90,000 | Rate: 1983.8 pages/sec
Pages inspected: 100,000 | Rate: 1986.0 pages/sec
Pages inspected: 110,000 | Rate: 2026.6 pages/sec
Pages inspected: 120,000 | Rate: 2058.5 pages/sec
Pages inspected: 130,000 | Rate: 2052.3 pages/sec
Pages inspected: 140,000 | Rate: 2067.8 pages/sec
Pages inspected: 150,000 | Rate: 2055.9 pages/sec
Pages inspected: 160,000 | Rate: 2020.0 pages/sec
Pages inspected: 170,000 | Rate: 2036.5 pages/sec
Pages inspected: 180,000 | Rate: 2041.3 pages/sec
Pages inspected: 190,000 | Rate: 2028.0 pages/sec
Pages inspected: 200,000 

In [ ]:
import bz2
import xml.etree.ElementTree as ET

samples = {
    "0": [],
    "104": []
}

TARGET_PER_NS = 10

with bz2.open(DUMP_FILE, "rb") as f:

    for event, elem in ET.iterparse(
        f,
        events=("end",)
    ):

        if not elem.tag.endswith("page"):
            continue

        ns = None
        title = ""
        text = ""

        for child in elem.iter():

            if child.tag.endswith("title"):
                title = child.text or ""

            elif child.tag.endswith("ns"):
                ns = child.text or ""

            elif child.tag.endswith("text"):
                text = child.text or ""

        if ns in samples and len(samples[ns]) < TARGET_PER_NS:

            samples[ns].append({
                "title": title,
                "text": text
            })

        elem.clear()

        if all(
            len(v) >= TARGET_PER_NS
            for v in samples.values()
        ):
            break


for ns, pages in samples.items():

    print("\n")
    print("=" * 80)
    print(f"NAMESPACE {ns} — {len(pages)} SAMPLES")
    print("=" * 80)

    for i, page in enumerate(pages, 1):

        print("\n" + "-" * 80)
        print(f"SAMPLE {i}")
        print("TITLE:", page["title"])
        print("TEXT:")
        print(page["text"][:1500])



NAMESPACE 0 — 10 SAMPLES

--------------------------------------------------------------------------------
SAMPLE 1
TITLE: మొదటి పేజీ
TEXT:
<templatestyles src="Template:Main Page/minerva.css" />
<!-- titlebox -->
<div style="font-size: 180%; border: none; margin:auto;text-align:center;">[[Wikisource:గురించి|వికీసోర్స్]] కు [[వికీసోర్స్:స్వాగతం|స్వాగతం!]]</div> 
{|style="width:100%; margin:0px; border-spacing:0; border: 1px solid #bfbfbf;" cellspacing='0' class="nomobile"
|-style="background-color:#fafafa;"
|style="width:10%; text-align:center;padding-left:1em; padding-right:1em;padding-bottom:1em; padding-top:1em;"|
[[దస్త్రం:Tewshomepageicon.png|middle|left|90px]] 
|style="width:60%; text-align:center; padding-right:5em;"|
<div style='padding:0.5em 0 0.75em 0;'>ఎవరైనా అభివృద్ధిపరచగల   [[w:నకలు_హక్కుల_చట్టం|స్వేచ్ఛానకలు హక్కులున్న]] రచనలుగల గ్రంథాలయము.
<div id="articlecount" style="font-size:90%;">&mdash; '''[[Special:Statistics|{{NUMBEROFARTICLES}}]]''' తెలుగు పాఠ్యపు ప్రధాన పేజీలు

In [ ]:
import os

RAW_CHUNK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/raw_chunks"
)

os.makedirs(RAW_CHUNK_DIR, exist_ok=True)

print("Raw chunk directory:")
print(RAW_CHUNK_DIR)

Raw chunk directory:
/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/wikisource/raw_chunks


In [ ]:
import bz2
import json
import os
import time
import xml.etree.ElementTree as ET

CHUNK_SIZE = 5_000

DUMP_FILE = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/"
    "tewikisource-20260801-pages-articles.xml.bz2"
)

RAW_CHUNK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/raw_chunks"
)

os.makedirs(RAW_CHUNK_DIR, exist_ok=True)


def get_text(page):
    """Extract page text from a MediaWiki XML <page> element."""
    for child in page.iter():
        if child.tag.endswith("text"):
            return child.text or ""
    return ""


def get_value(page, field):
    """Extract a simple XML field such as title or ns."""
    for child in page.iter():
        if child.tag.endswith(field):
            return child.text or ""
    return ""


def is_redirect(text):
    return text.lstrip().lower().startswith("#redirect")


# ------------------------------------------------------------
# Determine already completed chunks
# ------------------------------------------------------------

completed = set()

for filename in os.listdir(RAW_CHUNK_DIR):

    if not filename.startswith("raw_chunk_"):
        continue

    if not filename.endswith(".jsonl"):
        continue

    try:
        number = int(
            filename.replace(
                "raw_chunk_", ""
            ).replace(
                ".jsonl", ""
            )
        )

        completed.add(number)

    except ValueError:
        pass


print("Already completed chunks:", sorted(completed))


# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------

page_number = 0

kept = 0
removed_redirect = 0
removed_empty = 0
removed_namespace = 0

chunk_records = []

chunk_number = 0

start_time = time.time()


# ------------------------------------------------------------
# Read XML sequentially
# ------------------------------------------------------------

with bz2.open(DUMP_FILE, "rb") as f:

    for event, elem in ET.iterparse(
        f,
        events=("end",)
    ):

        if not elem.tag.endswith("page"):
            continue

        page_number += 1

        # ----------------------------------------------------
        # Extract fields
        # ----------------------------------------------------

        namespace = get_value(elem, "ns")
        title = get_value(elem, "title")
        text = get_text(elem)

        # ----------------------------------------------------
        # Decide whether page belongs to our corpus
        # ----------------------------------------------------

        keep = True

        if namespace not in {"0", "104"}:
            removed_namespace += 1
            keep = False

        elif is_redirect(text):
            removed_redirect += 1
            keep = False

        elif not text.strip():
            removed_empty += 1
            keep = False

        # ----------------------------------------------------
        # Save candidate
        # ----------------------------------------------------

        if keep:

            chunk_records.append({
                "title": title,
                "namespace": namespace,
                "text": text
            })

            kept += 1

        # ----------------------------------------------------
        # Chunk boundary
        #
        # IMPORTANT:
        # Chunk number is determined ONLY by page_number.
        # ----------------------------------------------------

        if page_number % CHUNK_SIZE == 0:

            chunk_number = (
                page_number // CHUNK_SIZE
            ) - 1

            output_file = os.path.join(
                RAW_CHUNK_DIR,
                f"raw_chunk_{chunk_number:05d}.jsonl"
            )

            # ------------------------------------------------
            # Never overwrite an existing completed chunk
            # ------------------------------------------------

            if chunk_number in completed:

                print(
                    f"SKIP existing chunk "
                    f"{chunk_number:05d} | "
                    f"Pages: {page_number:,}"
                )

            else:

                with open(
                    output_file,
                    "w",
                    encoding="utf-8"
                ) as out:

                    for record in chunk_records:

                        out.write(
                            json.dumps(
                                record,
                                ensure_ascii=False
                            )
                            + "\n"
                        )

                completed.add(chunk_number)

                elapsed = time.time() - start_time

                print(
                    f"CHECKPOINT {chunk_number:05d} SAVED | "
                    f"Pages: {page_number:,} | "
                    f"Records: {len(chunk_records):,} | "
                    f"Kept total: {kept:,} | "
                    f"Rate: "
                    f"{page_number / elapsed:.1f} pages/sec"
                )

            chunk_records = []

        # ----------------------------------------------------
        # Free XML memory
        # ----------------------------------------------------

        elem.clear()


# ------------------------------------------------------------
# Save final partial chunk
# ------------------------------------------------------------

if chunk_records:

    chunk_number = (
        page_number // CHUNK_SIZE
    )

    output_file = os.path.join(
        RAW_CHUNK_DIR,
        f"raw_chunk_{chunk_number:05d}.jsonl"
    )

    if chunk_number not in completed:

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as out:

            for record in chunk_records:

                out.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    )
                    + "\n"
                )

        print(
            f"FINAL CHECKPOINT {chunk_number:05d} SAVED | "
            f"Pages: {page_number:,} | "
            f"Records: {len(chunk_records):,}"
        )


print()
print("=" * 70)
print("RAW WIKISOURCE EXTRACTION COMPLETE")
print("=" * 70)

print("Pages processed:", page_number)
print("Kept:", kept)
print("Removed namespace:", removed_namespace)
print("Removed redirects:", removed_redirect)
print("Removed empty:", removed_empty)

Already completed chunks: []
CHECKPOINT 00000 SAVED | Pages: 5,000 | Records: 4,855 | Kept total: 4,855 | Rate: 1126.6 pages/sec
CHECKPOINT 00001 SAVED | Pages: 10,000 | Records: 4,535 | Kept total: 9,390 | Rate: 1113.6 pages/sec
CHECKPOINT 00002 SAVED | Pages: 15,000 | Records: 4,663 | Kept total: 14,053 | Rate: 907.1 pages/sec
CHECKPOINT 00003 SAVED | Pages: 20,000 | Records: 4,888 | Kept total: 18,941 | Rate: 1041.6 pages/sec
CHECKPOINT 00004 SAVED | Pages: 25,000 | Records: 4,903 | Kept total: 23,844 | Rate: 1173.9 pages/sec
CHECKPOINT 00005 SAVED | Pages: 30,000 | Records: 4,697 | Kept total: 28,541 | Rate: 1278.3 pages/sec
CHECKPOINT 00006 SAVED | Pages: 35,000 | Records: 4,881 | Kept total: 33,422 | Rate: 1317.3 pages/sec
CHECKPOINT 00007 SAVED | Pages: 40,000 | Records: 4,782 | Kept total: 38,204 | Rate: 1349.6 pages/sec
CHECKPOINT 00008 SAVED | Pages: 45,000 | Records: 4,959 | Kept total: 43,163 | Rate: 1361.8 pages/sec
CHECKPOINT 00009 SAVED | Pages: 50,000 | Records: 4,980 |

In [ ]:
import glob
import json
import os

files = sorted(
    glob.glob(
        RAW_CHUNK_DIR + "/raw_chunk_*.jsonl"
    )
)

print("Number of chunks:", len(files))

total_records = 0

for path in files:

    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1

    total_records += count

    print(
        os.path.basename(path),
        "| records:", count,
        "| size:",
        round(os.path.getsize(path) / (1024**2), 2),
        "MB"
    )

print()
print("TOTAL RECORDS:", total_records)

Number of chunks: 42
raw_chunk_00000.jsonl | records: 4855 | size: 31.55 MB
raw_chunk_00001.jsonl | records: 4535 | size: 28.84 MB
raw_chunk_00002.jsonl | records: 4663 | size: 42.1 MB
raw_chunk_00003.jsonl | records: 4888 | size: 15.13 MB
raw_chunk_00004.jsonl | records: 4903 | size: 12.61 MB
raw_chunk_00005.jsonl | records: 4697 | size: 10.31 MB
raw_chunk_00006.jsonl | records: 4881 | size: 14.13 MB
raw_chunk_00007.jsonl | records: 4782 | size: 15.27 MB
raw_chunk_00008.jsonl | records: 4959 | size: 15.49 MB
raw_chunk_00009.jsonl | records: 4980 | size: 19.79 MB
raw_chunk_00010.jsonl | records: 4757 | size: 14.79 MB
raw_chunk_00011.jsonl | records: 4793 | size: 14.05 MB
raw_chunk_00012.jsonl | records: 4687 | size: 13.01 MB
raw_chunk_00013.jsonl | records: 4878 | size: 16.45 MB
raw_chunk_00014.jsonl | records: 4751 | size: 15.33 MB
raw_chunk_00015.jsonl | records: 4895 | size: 14.51 MB
raw_chunk_00016.jsonl | records: 4869 | size: 13.1 MB
raw_chunk_00017.jsonl | records: 4884 | size: 

In [ ]:
import json
import glob

namespace_counts = {
    "0": 0,
    "104": 0
}

examples = {
    "0": [],
    "104": []
}

for path in sorted(
    glob.glob(RAW_CHUNK_DIR + "/raw_chunk_*.jsonl")
):

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            ns = row["namespace"]

            namespace_counts[ns] = (
                namespace_counts.get(ns, 0) + 1
            )

            if len(examples.get(ns, [])) < 5:
                examples.setdefault(ns, []).append(row)


print("Namespace counts:")
for ns, count in namespace_counts.items():
    print(ns, count)


for ns in ["0", "104"]:

    print("\n" + "=" * 80)
    print("NAMESPACE:", ns)
    print("=" * 80)

    for i, row in enumerate(examples[ns], 1):

        print("\n---", i, "---")
        print("TITLE:", row["title"])
        print("TEXT:", row["text"][:1000])

Namespace counts:
0 21727
104 180300

NAMESPACE: 0

--- 1 ---
TITLE: మొదటి పేజీ
TEXT: <templatestyles src="Template:Main Page/minerva.css" />
<!-- titlebox -->
<div style="font-size: 180%; border: none; margin:auto;text-align:center;">[[Wikisource:గురించి|వికీసోర్స్]] కు [[వికీసోర్స్:స్వాగతం|స్వాగతం!]]</div> 
{|style="width:100%; margin:0px; border-spacing:0; border: 1px solid #bfbfbf;" cellspacing='0' class="nomobile"
|-style="background-color:#fafafa;"
|style="width:10%; text-align:center;padding-left:1em; padding-right:1em;padding-bottom:1em; padding-top:1em;"|
[[దస్త్రం:Tewshomepageicon.png|middle|left|90px]] 
|style="width:60%; text-align:center; padding-right:5em;"|
<div style='padding:0.5em 0 0.75em 0;'>ఎవరైనా అభివృద్ధిపరచగల   [[w:నకలు_హక్కుల_చట్టం|స్వేచ్ఛానకలు హక్కులున్న]] రచనలుగల గ్రంథాలయము.
<div id="articlecount" style="font-size:90%;">&mdash; '''[[Special:Statistics|{{NUMBEROFARTICLES}}]]''' తెలుగు పాఠ్యపు ప్రధాన పేజీలు,
{{PAGESINCATEGORY:దింపుకొనదగిన పుస్తకాలు}} [[:వర్గం:ది

In [ ]:
import re
import json
import glob
import mwparserfromhell
from collections import Counter

RAW_CHUNK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/raw_chunks"
)

TELUGU_RE = re.compile(r'[\u0C00-\u0C7F]')


# ------------------------------------------------------------
# PageQuality
# ------------------------------------------------------------

def extract_page_quality(text):
    matches = re.findall(
        r'\{\{\s*PageQuality\s*\|\s*([0-9]+)\s*%\s*\}\}',
        text,
        flags=re.IGNORECASE
    )

    if not matches:
        return None

    return int(matches[-1])


# ------------------------------------------------------------
# Telugu character count
# ------------------------------------------------------------

def telugu_count(text):
    return sum(
        1 for c in text
        if TELUGU_RE.match(c)
    )


# ------------------------------------------------------------
# V3 cleaner
# ------------------------------------------------------------

def clean_wikisource_text_v3(text):

    # Remove comments
    text = re.sub(
        r'<!--.*?-->',
        ' ',
        text,
        flags=re.DOTALL
    )

    # Remove noinclude/includeonly blocks
    text = re.sub(
        r'<noinclude>.*?</noinclude>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    text = re.sub(
        r'<includeonly>.*?</includeonly>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # Remove page quality metadata
    text = re.sub(
        r'\{\{\s*PageQuality\s*\|\s*[0-9]+\s*%\s*\}\}',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'<pagequality[^>]*/?>',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Keep contents inside poem/pre
    text = re.sub(
        r'<(?:poem|pre)[^>]*>',
        '',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'</(?:poem|pre)>',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Parse MediaWiki
    try:
        code = mwparserfromhell.parse(text)
    except Exception:
        return None

    # Remove templates
    for template in list(
        code.filter_templates(recursive=True)
    ):
        try:
            code.remove(template)
        except ValueError:
            pass

    # Convert wiki links
    for link in list(
        code.filter_wikilinks(recursive=True)
    ):
        try:
            if link.text is not None:
                replacement = str(link.text)
            else:
                replacement = str(link.title)

            code.replace(link, replacement)

        except ValueError:
            pass

    text = str(code)

    # Remove MediaWiki tables
    text = re.sub(
        r'\{\|.*?\|\}',
        ' ',
        text,
        flags=re.DOTALL
    )

    # Remove files/images
    text = re.sub(
        r'\[\[(?:File|దస్త్రం):.*?\]\]',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove references
    text = re.sub(
        r'<ref\b[^>]*>.*?</ref>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    text = re.sub(
        r'<ref\b[^>]*/>',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove remaining HTML
    text = re.sub(
        r'<[^>]+>',
        ' ',
        text
    )

    # Remove category / metadata lines
    text = re.sub(
        r'^\s*(?:Category|వర్గం)\s*:[^\n]*$',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE
    )

    # Remove raw category links that survived
    text = re.sub(
        r'\[\[(?:Category|వర్గం):.*?\]\]',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove formatting
    text = re.sub(
        r"'{2,5}",
        '',
        text
    )

    # Remove table remnants
    text = re.sub(
        r'^\s*[\|\!].*$',
        '',
        text,
        flags=re.MULTILINE
    )

    # Clean excessive whitespace
    text = re.sub(
        r'[ \t]+',
        ' ',
        text
    )

    text = re.sub(
        r'\n\s*\n\s*\n+',
        '\n\n',
        text
    )

    return text.strip()


# ------------------------------------------------------------
# V3 index/navigation detector
# ------------------------------------------------------------

def looks_like_index_v3(title, cleaned):

    lines = [
        x.strip()
        for x in cleaned.splitlines()
        if x.strip()
    ]

    if not lines:
        return True

    # Very short content
    if len(cleaned) < 100:
        return True

    # Don't classify normal prose just because it has bullets.
    list_lines = sum(
        line.startswith(("#", "*", ";"))
        for line in lines
    )

    if len(lines) >= 5:

        list_ratio = list_lines / len(lines)

        if list_ratio >= 0.80:
            return True

    # Strong index/catalogue indicators in title.
    index_patterns = [
        r'కృతులు\s*[అ-హ]',
        r'కృతుల\s*సూచిక',
        r'సూచిక',
        r'అక్షరక్రమ',
        r'జాబితా',
        r'contents',
        r'index'
    ]

    for pattern in index_patterns:

        if re.search(
            pattern,
            title,
            flags=re.IGNORECASE
        ):
            return True

    return False


# ============================================================
# V3 TEST
# ============================================================

files = sorted(
    glob.glob(
        RAW_CHUNK_DIR + "/raw_chunk_*.jsonl"
    )
)[:4]

stats = Counter()

examples = {
    "kept": [],
    "index": [],
    "short": [],
    "empty": [],
    "parser_error": [],
    "low_telugu_but_kept": []
}

for path in files:

    print("Processing:", path.split("/")[-1])

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            stats["raw"] += 1

            title = row["title"]
            namespace = row["namespace"]
            raw = row["text"]

            # ------------------------------------------------
            # V3: namespace 0 only
            # ------------------------------------------------

            if namespace != "0":
                stats["namespace_104_removed"] += 1
                continue

            stats["namespace_0"] += 1

            quality = extract_page_quality(raw)

            cleaned = clean_wikisource_text_v3(raw)

            if cleaned is None:

                stats["parser_error"] += 1

                if len(examples["parser_error"]) < 5:
                    examples["parser_error"].append(title)

                continue

            if not cleaned:

                stats["empty"] += 1

                if len(examples["empty"]) < 10:
                    examples["empty"].append(title)

                continue

            chars = len(
                re.sub(
                    r'\s+',
                    '',
                    cleaned
                )
            )

            telugu_chars = telugu_count(cleaned)

            # ------------------------------------------------
            # Remove genuinely tiny documents
            # ------------------------------------------------

            if chars < 100:

                stats["short"] += 1

                if len(examples["short"]) < 10:
                    examples["short"].append(
                        (title, cleaned[:500])
                    )

                continue

            # ------------------------------------------------
            # Remove obvious index pages
            # ------------------------------------------------

            if looks_like_index_v3(
                title,
                cleaned
            ):

                stats["index"] += 1

                if len(examples["index"]) < 15:
                    examples["index"].append(
                        (title, cleaned[:700])
                    )

                continue

            # ------------------------------------------------
            # Candidate
            # ------------------------------------------------

            stats["kept"] += 1

            if (
                telugu_chars < 50
                and len(examples["low_telugu_but_kept"]) < 10
            ):
                examples["low_telugu_but_kept"].append(
                    (
                        title,
                        telugu_chars,
                        cleaned[:700]
                    )
                )

            if len(examples["kept"]) < 15:
                examples["kept"].append(
                    (
                        title,
                        quality,
                        telugu_chars,
                        cleaned[:1000]
                    )
                )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("V3 DIAGNOSTIC COMPLETE")
print("=" * 80)

for key, value in stats.items():
    print(f"{key}: {value:,}")


print()
print("=" * 80)
print("KEPT EXAMPLES")
print("=" * 80)

for title, quality, telugu_chars, text in examples["kept"]:

    print("\nTITLE:", title)
    print("PageQuality:", quality)
    print("Telugu chars:", telugu_chars)
    print("TEXT:")
    print(text)


print()
print("=" * 80)
print("INDEX REMOVED")
print("=" * 80)

for title, text in examples["index"]:

    print("\nTITLE:", title)
    print("TEXT:", text)


print()
print("=" * 80)
print("LOW-TELUGU BUT KEPT")
print("=" * 80)

for title, count, text in examples["low_telugu_but_kept"]:

    print("\nTITLE:", title)
    print("Telugu chars:", count)
    print("TEXT:", text)

Processing: raw_chunk_00000.jsonl
Processing: raw_chunk_00001.jsonl
Processing: raw_chunk_00002.jsonl
Processing: raw_chunk_00003.jsonl

V3 DIAGNOSTIC COMPLETE
raw: 18,941
namespace_0: 10,087
short: 805
kept: 8,260
index: 215
empty: 807
namespace_104_removed: 8,854

KEPT EXAMPLES

TITLE: ఆంధ్ర మాత
PageQuality: None
Telugu chars: 2901
TEXT:
తిక్కనాకర్యుని వాణి త్రుప్తినంద

గజ్జెలందెల కాలు గడలించి మన్న బొ
బ్బిలి కోట దొలికోడి పలికి కులుక

నాడువీధులందు రత్నము లమ్ముకొన్నట్టి
తెలుగుల సంపత్తి తలపుకెక్క

ఖండాంతరముల బంగారు పూజలు గొన్న
కలికి పోగరమేను పులకరింప||

మూడు కొండల శివలింగములు రహింప
నాలశింపక రాష్ట్ర సింహాసనమున
నాధివశింపుము జయము నీయందు కలదు
దివ్యదాసయ సుమవల్లి తెలుగు తల్లి 

గణ గాంధేయ శక ప్రకాశలాన మింకన్ శోక కాత్యంత ప్రా
క్తనపుంజీకటి చిందులాడెడు విముగ్ధ ప్రాంతముల్ నిద్ర మే
ల్కొన శంఖారవ మాచరింపు మిక నుగ్గుంబాలలో గాంతమం
దిన విధ్వాంశులగన్న వీర జనయిత్రీ ఆంధ్రరత్నాక్షితీ 

కులముల్ గొమ్ములాటొడ గుమ్ముకొని చిక్కుల్ పెక్కు శృష్టించు పె
ద్దల కాలాలు గతించి పోయినవి స్వాతంత్య్రంబు సిధ్ధించె రా
జుల సింహ

In [ ]:
import re
import json
import glob
from collections import Counter
import mwparserfromhell

RAW_CHUNK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/raw_chunks"
)

TELUGU_RE = re.compile(r'[\u0C00-\u0C7F]')


def extract_page_quality(text):
    m = re.findall(
        r'\{\{\s*PageQuality\s*\|\s*([0-9]+)\s*%\s*\}\}',
        text,
        flags=re.I
    )
    return int(m[-1]) if m else None


def telugu_count(text):
    return sum(bool(TELUGU_RE.match(c)) for c in text)


def clean_wikisource_text_v4(text):

    # Comments
    text = re.sub(
        r'<!--.*?-->', ' ',
        text, flags=re.S
    )

    # noinclude/includeonly
    text = re.sub(
        r'<noinclude>.*?</noinclude>', ' ',
        text, flags=re.S | re.I
    )
    text = re.sub(
        r'<includeonly>.*?</includeonly>', ' ',
        text, flags=re.S | re.I
    )

    # Page quality metadata
    text = re.sub(
        r'\{\{\s*PageQuality\s*\|\s*[0-9]+\s*%\s*\}\}',
        ' ', text, flags=re.I
    )
    text = re.sub(
        r'<pagequality[^>]*/?>',
        ' ', text, flags=re.I
    )

    # Preserve contents of poem/pre
    text = re.sub(
        r'<(?:poem|pre)[^>]*>', '',
        text, flags=re.I
    )
    text = re.sub(
        r'</(?:poem|pre)>', '',
        text, flags=re.I
    )

    try:
        code = mwparserfromhell.parse(text)
    except Exception:
        return None

    # Templates
    for template in list(code.filter_templates(recursive=True)):
        try:
            code.remove(template)
        except ValueError:
            pass

    # Wiki links -> visible text
    for link in list(code.filter_wikilinks(recursive=True)):
        try:
            replacement = (
                str(link.text)
                if link.text is not None
                else str(link.title)
            )
            code.replace(link, replacement)
        except ValueError:
            pass

    text = str(code)

    # Tables
    text = re.sub(
        r'\{\|.*?\|\}',
        ' ', text, flags=re.S
    )

    # Files/images
    text = re.sub(
        r'\[\[(?:File|దస్త్రం):.*?\]\]',
        ' ', text, flags=re.I
    )

    # References
    text = re.sub(
        r'<ref\b[^>]*>.*?</ref>',
        ' ', text, flags=re.S | re.I
    )
    text = re.sub(
        r'<ref\b[^>]*/>',
        ' ', text, flags=re.I
    )

    # HTML
    text = re.sub(r'<[^>]+>', ' ', text)

    # Categories that survived parsing
    text = re.sub(
        r'^\s*(?:Category|వర్గం)\s*:[^\n]*$',
        '', text, flags=re.M | re.I
    )

    # Formatting
    text = re.sub(r"'{2,5}", '', text)

    # Table remnants
    text = re.sub(
        r'^\s*[\|\!].*$',
        '', text, flags=re.M
    )

    # Whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)

    return text.strip()


def looks_like_index_v4(title, text):

    lines = [
        x.strip()
        for x in text.splitlines()
        if x.strip()
    ]

    if not lines:
        return True

    # Very short
    if len(text) < 100:
        return True

    # Strong list/index signal
    list_lines = sum(
        line.startswith(("#", "*", ";"))
        for line in lines
    )

    if len(lines) >= 5 and list_lines / len(lines) >= 0.80:
        return True

    # Strong title indicators only
    patterns = [
        r'కృతులు\s*[అ-హ]',
        r'కృతుల\s*సూచిక',
        r'సూచిక',
        r'అక్షరక్రమ',
        r'జాబితా',
        r'contents',
        r'index'
    ]

    return any(
        re.search(p, title, flags=re.I)
        for p in patterns
    )


# ============================================================
# V4 TEST — FIRST 4 CHUNKS ONLY
# ============================================================

files = sorted(
    glob.glob(RAW_CHUNK_DIR + "/raw_chunk_*.jsonl")
)[:4]

stats = Counter()

examples = {
    "kept": [],
    "short": [],
    "index": [],
    "low_telugu": [],
    "empty": [],
}

telugu_distribution = Counter()

for path in files:

    print("Processing:", path.split("/")[-1])

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            stats["raw"] += 1

            # Namespace 0 only
            if row["namespace"] != "0":
                stats["namespace_104_removed"] += 1
                continue

            stats["namespace_0"] += 1

            title = row["title"]
            raw = row["text"]

            cleaned = clean_wikisource_text_v4(raw)

            if not cleaned:
                stats["empty"] += 1

                if len(examples["empty"]) < 10:
                    examples["empty"].append(title)

                continue

            chars = len(re.sub(r'\s+', '', cleaned))
            te_chars = telugu_count(cleaned)

            # Record distribution
            if te_chars < 10:
                bucket = "<10"
            elif te_chars < 25:
                bucket = "10-24"
            elif te_chars < 50:
                bucket = "25-49"
            elif te_chars < 100:
                bucket = "50-99"
            elif te_chars < 250:
                bucket = "100-249"
            elif te_chars < 500:
                bucket = "250-499"
            elif te_chars < 1000:
                bucket = "500-999"
            else:
                bucket = "1000+"

            telugu_distribution[bucket] += 1

            # Short
            if chars < 100:
                stats["short"] += 1

                if len(examples["short"]) < 10:
                    examples["short"].append(
                        (title, cleaned[:500])
                    )

                continue

            # Index/navigation
            if looks_like_index_v4(title, cleaned):
                stats["index"] += 1

                if len(examples["index"]) < 15:
                    examples["index"].append(
                        (title, cleaned[:700])
                    )

                continue

            # V4: minimum 100 Telugu characters
            if te_chars < 100:
                stats["low_telugu"] += 1

                if len(examples["low_telugu"]) < 15:
                    examples["low_telugu"].append(
                        (title, te_chars, cleaned[:700])
                    )

                continue

            # KEEP
            stats["kept"] += 1

            if len(examples["kept"]) < 20:
                examples["kept"].append(
                    (
                        title,
                        extract_page_quality(raw),
                        te_chars,
                        cleaned[:1200]
                    )
                )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("V4 DIAGNOSTIC COMPLETE")
print("=" * 80)

print("\nStatistics:")

for key, value in stats.items():
    print(f"{key}: {value:,}")

print("\nTelugu-character distribution:")

for bucket in [
    "<10", "10-24", "25-49", "50-99",
    "100-249", "250-499", "500-999", "1000+"
]:
    print(
        f"{bucket:>8}: "
        f"{telugu_distribution[bucket]:,}"
    )


print("\n" + "=" * 80)
print("KEPT EXAMPLES")
print("=" * 80)

for title, quality, te_chars, text in examples["kept"]:
    print("\nTITLE:", title)
    print("PageQuality:", quality)
    print("Telugu chars:", te_chars)
    print("TEXT:")
    print(text)


print("\n" + "=" * 80)
print("LOW-TELUGU REMOVED")
print("=" * 80)

for title, te_chars, text in examples["low_telugu"]:
    print("\nTITLE:", title)
    print("Telugu chars:", te_chars)
    print("TEXT:", text)


print("\n" + "=" * 80)
print("INDEX REMOVED")
print("=" * 80)

for title, text in examples["index"]:
    print("\nTITLE:", title)
    print("TEXT:", text)

Processing: raw_chunk_00000.jsonl
Processing: raw_chunk_00001.jsonl
Processing: raw_chunk_00002.jsonl
Processing: raw_chunk_00003.jsonl

V4 DIAGNOSTIC COMPLETE

Statistics:
raw: 18,941
namespace_0: 10,087
short: 805
kept: 8,210
index: 215
empty: 807
namespace_104_removed: 8,854
low_telugu: 50

Telugu-character distribution:
     <10: 77
   10-24: 89
   25-49: 348
   50-99: 353
 100-249: 337
 250-499: 2,137
 500-999: 894
   1000+: 5,045

KEPT EXAMPLES

TITLE: ఆంధ్ర మాత
PageQuality: None
Telugu chars: 2901
TEXT:
తిక్కనాకర్యుని వాణి త్రుప్తినంద

గజ్జెలందెల కాలు గడలించి మన్న బొ
బ్బిలి కోట దొలికోడి పలికి కులుక

నాడువీధులందు రత్నము లమ్ముకొన్నట్టి
తెలుగుల సంపత్తి తలపుకెక్క

ఖండాంతరముల బంగారు పూజలు గొన్న
కలికి పోగరమేను పులకరింప||

మూడు కొండల శివలింగములు రహింప
నాలశింపక రాష్ట్ర సింహాసనమున
నాధివశింపుము జయము నీయందు కలదు
దివ్యదాసయ సుమవల్లి తెలుగు తల్లి 

గణ గాంధేయ శక ప్రకాశలాన మింకన్ శోక కాత్యంత ప్రా
క్తనపుంజీకటి చిందులాడెడు విముగ్ధ ప్రాంతముల్ నిద్ర మే
ల్కొన శంఖారవ మాచరింపు మిక నుగ్గుంబాలలో గాంతమం


In [ ]:
import os
import re
import json
import glob
import time
import mwparserfromhell
from collections import Counter

# ============================================================
# PATHS
# ============================================================

RAW_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/wikisource/raw_chunks"
)

OUT_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_checkpoints"
)

MANIFEST_PATH = os.path.join(OUT_DIR, "manifest.json")

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================

CHUNK_SIZE = 5000
MIN_TELUGU_CHARS = 50
MIN_TEXT_CHARS = 100

TELUGU_RE = re.compile(r'[\u0C00-\u0C7F]')


# ============================================================
# HELPERS
# ============================================================

def telugu_count(text):
    return sum(
        1 for c in text
        if TELUGU_RE.match(c)
    )


def extract_page_quality(text):
    matches = re.findall(
        r'\{\{\s*PageQuality\s*\|\s*([0-9]+)\s*%\s*\}\}',
        text,
        flags=re.IGNORECASE
    )

    return int(matches[-1]) if matches else None


# ============================================================
# V4 CLEANER
# ============================================================

def clean_wikisource_text_v4(text):

    # Comments
    text = re.sub(
        r'<!--.*?-->',
        ' ',
        text,
        flags=re.DOTALL
    )

    # noinclude / includeonly
    text = re.sub(
        r'<noinclude>.*?</noinclude>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    text = re.sub(
        r'<includeonly>.*?</includeonly>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    # PageQuality
    text = re.sub(
        r'\{\{\s*PageQuality\s*\|\s*[0-9]+\s*%\s*\}\}',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'<pagequality[^>]*/?>',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # poem / pre tags
    text = re.sub(
        r'<(?:poem|pre)[^>]*>',
        '',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'</(?:poem|pre)>',
        '',
        text,
        flags=re.IGNORECASE
    )

    # MediaWiki parser
    try:
        code = mwparserfromhell.parse(text)
    except Exception:
        return None

    # Remove templates
    for template in list(
        code.filter_templates(recursive=True)
    ):
        try:
            code.remove(template)
        except ValueError:
            pass

    # Convert wikilinks to visible text
    for link in list(
        code.filter_wikilinks(recursive=True)
    ):
        try:

            if link.text is not None:
                replacement = str(link.text)
            else:
                replacement = str(link.title)

            code.replace(link, replacement)

        except ValueError:
            pass

    text = str(code)

    # MediaWiki tables
    text = re.sub(
        r'\{\|.*?\|\}',
        ' ',
        text,
        flags=re.DOTALL
    )

    # Files/images
    text = re.sub(
        r'\[\[(?:File|దస్త్రం):.*?\]\]',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # References
    text = re.sub(
        r'<ref\b[^>]*>.*?</ref>',
        ' ',
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    text = re.sub(
        r'<ref\b[^>]*/>',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remaining HTML
    text = re.sub(
        r'<[^>]+>',
        ' ',
        text
    )

    # Category / metadata lines
    text = re.sub(
        r'^\s*(?:Category|వర్గం)\s*:[^\n]*$',
        '',
        text,
        flags=re.MULTILINE | re.IGNORECASE
    )

    # Remaining category links
    text = re.sub(
        r'\[\[(?:Category|వర్గం):.*?\]\]',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Formatting
    text = re.sub(
        r"'{2,5}",
        '',
        text
    )

    # Table remnants
    text = re.sub(
        r'^\s*[\|\!].*$',
        '',
        text,
        flags=re.MULTILINE
    )

    # Whitespace
    text = re.sub(
        r'[ \t]+',
        ' ',
        text
    )

    text = re.sub(
        r'\n\s*\n\s*\n+',
        '\n\n',
        text
    )

    return text.strip()


# ============================================================
# INDEX / NAVIGATION DETECTOR
# ============================================================

def looks_like_index_v4(title, text):

    lines = [
        x.strip()
        for x in text.splitlines()
        if x.strip()
    ]

    if not lines:
        return True

    # Very short
    if len(text) < MIN_TEXT_CHARS:
        return True

    # Extremely list-heavy pages
    list_lines = sum(
        line.startswith(("#", "*", ";"))
        for line in lines
    )

    if len(lines) >= 5:

        if list_lines / len(lines) >= 0.80:
            return True

    # Strong index indicators
    patterns = [
        r'కృతులు\s*[అ-హ]',
        r'కృతుల\s*సూచిక',
        r'సూచిక',
        r'అక్షరక్రమ',
        r'జాబితా',
        r'contents',
        r'index'
    ]

    for pattern in patterns:

        if re.search(
            pattern,
            title,
            flags=re.IGNORECASE
        ):
            return True

    return False


# ============================================================
# RAW CHUNKS
# ============================================================

raw_files = sorted(
    glob.glob(
        os.path.join(
            RAW_DIR,
            "raw_chunk_*.jsonl"
        )
    )
)

print("Raw chunk files:", len(raw_files))

for f in raw_files:
    print(
        os.path.basename(f),
        f"{os.path.getsize(f) / (1024**2):.2f} MB"
    )


# ============================================================
# MANIFEST
# ============================================================

if os.path.exists(MANIFEST_PATH):

    with open(
        MANIFEST_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(f)

else:

    manifest = {
        "version": "v4",
        "chunk_size": CHUNK_SIZE,
        "min_text_chars": MIN_TEXT_CHARS,
        "min_telugu_chars": MIN_TELUGU_CHARS,
        "completed_chunks": [],
        "stats": {}
    }


completed = set(
    manifest.get(
        "completed_chunks",
        []
    )
)

print(
    "\nAlready completed chunks:",
    sorted(completed)
)


# ============================================================
# GLOBAL STATS
# ============================================================

stats = Counter(
    manifest.get(
        "stats",
        {}
    )
)

total_start = time.time()


# ============================================================
# PROCESS RAW CHUNKS
# ============================================================

for chunk_index, raw_path in enumerate(raw_files):

    if chunk_index in completed:

        print(
            f"\nSKIPPING chunk {chunk_index:05d} "
            "(already completed)"
        )

        continue

    print()
    print("=" * 80)
    print(
        f"PROCESSING CHUNK {chunk_index:05d}"
    )
    print("=" * 80)

    chunk_records = []

    pages_in_chunk = 0
    start_time = time.time()

    with open(
        raw_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            stats["raw_records"] += 1
            pages_in_chunk += 1

            namespace = row.get(
                "namespace"
            )

            # ------------------------------------------------
            # Namespace 0 only
            # ------------------------------------------------

            if str(namespace) != "0":

                stats["removed_namespace"] += 1
                continue

            stats["namespace_0"] += 1

            title = row.get(
                "title",
                ""
            )

            raw_text = row.get(
                "text",
                ""
            )

            # ------------------------------------------------
            # Clean
            # ------------------------------------------------

            cleaned = clean_wikisource_text_v4(
                raw_text
            )

            if cleaned is None:

                stats["parser_errors"] += 1
                continue

            if not cleaned:

                stats["removed_empty"] += 1
                continue

            # ------------------------------------------------
            # Character checks
            # ------------------------------------------------

            compact = re.sub(
                r'\s+',
                '',
                cleaned
            )

            char_count = len(compact)

            if char_count < MIN_TEXT_CHARS:

                stats["removed_short"] += 1
                continue

            # ------------------------------------------------
            # Index/navigation
            # ------------------------------------------------

            if looks_like_index_v4(
                title,
                cleaned
            ):

                stats["removed_index"] += 1
                continue

            # ------------------------------------------------
            # Telugu threshold
            # ------------------------------------------------

            te_chars = telugu_count(
                cleaned
            )

            if te_chars < MIN_TELUGU_CHARS:

                stats["removed_low_telugu"] += 1
                continue

            # ------------------------------------------------
            # KEEP
            # ------------------------------------------------

            doc = {
                "doc_id": row.get(
                    "doc_id"
                ),
                "title": title,
                "text": cleaned,
                "source": "telugu_wikisource",
                "namespace": 0,
                "page_quality": extract_page_quality(
                    raw_text
                ),
                "char_count": char_count,
                "telugu_chars": te_chars
            }

            chunk_records.append(doc)

            stats["kept"] += 1

            # Progress every 1,000 raw records
            if pages_in_chunk % 1000 == 0:

                elapsed = (
                    time.time() -
                    start_time
                )

                rate = (
                    pages_in_chunk / elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"  {pages_in_chunk:,} raw records | "
                    f"kept in chunk: {len(chunk_records):,} | "
                    f"rate: {rate:.1f}/sec"
                )

    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    output_path = os.path.join(
        OUT_DIR,
        f"wikisource_v4_chunk_{chunk_index:05d}.jsonl"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as out:

        for doc in chunk_records:

            out.write(
                json.dumps(
                    doc,
                    ensure_ascii=False
                ) + "\n"
            )

    completed.add(
        chunk_index
    )

    manifest["completed_chunks"] = sorted(
        completed
    )

    manifest["stats"] = dict(
        stats
    )

    manifest["last_completed_chunk"] = (
        chunk_index
    )

    manifest["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(
        MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            manifest,
            f,
            ensure_ascii=False,
            indent=2
        )

    elapsed = time.time() - start_time

    print()
    print(
        f"CHECKPOINT {chunk_index:05d} SAVED"
    )
    print(
        f"Raw records: {pages_in_chunk:,}"
    )
    print(
        f"Records kept: {len(chunk_records):,}"
    )
    print(
        f"File: {os.path.basename(output_path)}"
    )
    print(
        f"Rate: {pages_in_chunk / elapsed:.1f} records/sec"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

total_elapsed = time.time() - total_start

print()
print("=" * 80)
print("FULL WIKISOURCE V4 EXTRACTION COMPLETE")
print("=" * 80)

print(
    f"Raw records:       {stats['raw_records']:,}"
)

print(
    f"Namespace 0:       {stats['namespace_0']:,}"
)

print(
    f"Kept:              {stats['kept']:,}"
)

print(
    f"Removed namespace: {stats['removed_namespace']:,}"
)

print(
    f"Removed empty:     {stats['removed_empty']:,}"
)

print(
    f"Removed short:     {stats['removed_short']:,}"
)

print(
    f"Removed index:     {stats['removed_index']:,}"
)

print(
    f"Removed low Telugu:{stats['removed_low_telugu']:,}"
)

print(
    f"Parser errors:     {stats['parser_errors']:,}"
)

print(
    f"Elapsed:           {total_elapsed / 60:.2f} minutes"
)

print(
    f"Output directory:  {OUT_DIR}"
)

print(
    f"Manifest:          {MANIFEST_PATH}"
)

Raw chunk files: 42
raw_chunk_00000.jsonl 31.55 MB
raw_chunk_00001.jsonl 28.84 MB
raw_chunk_00002.jsonl 42.10 MB
raw_chunk_00003.jsonl 15.13 MB
raw_chunk_00004.jsonl 12.61 MB
raw_chunk_00005.jsonl 10.31 MB
raw_chunk_00006.jsonl 14.13 MB
raw_chunk_00007.jsonl 15.27 MB
raw_chunk_00008.jsonl 15.49 MB
raw_chunk_00009.jsonl 19.79 MB
raw_chunk_00010.jsonl 14.79 MB
raw_chunk_00011.jsonl 14.05 MB
raw_chunk_00012.jsonl 13.01 MB
raw_chunk_00013.jsonl 16.45 MB
raw_chunk_00014.jsonl 15.33 MB
raw_chunk_00015.jsonl 14.51 MB
raw_chunk_00016.jsonl 13.10 MB
raw_chunk_00017.jsonl 12.80 MB
raw_chunk_00018.jsonl 13.05 MB
raw_chunk_00019.jsonl 14.59 MB
raw_chunk_00020.jsonl 14.31 MB
raw_chunk_00021.jsonl 13.94 MB
raw_chunk_00022.jsonl 16.38 MB
raw_chunk_00023.jsonl 13.05 MB
raw_chunk_00024.jsonl 16.32 MB
raw_chunk_00025.jsonl 15.64 MB
raw_chunk_00026.jsonl 12.79 MB
raw_chunk_00027.jsonl 17.67 MB
raw_chunk_00028.jsonl 22.83 MB
raw_chunk_00029.jsonl 19.06 MB
raw_chunk_00030.jsonl 18.01 MB
raw_chunk_00031.jso

In [ ]:
import os
import json
import glob
import re
from collections import Counter, defaultdict

V4_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_checkpoints"
)

files = sorted(
    glob.glob(
        os.path.join(
            V4_DIR,
            "wikisource_v4_chunk_*.jsonl"
        )
    )
)

print("=" * 80)
print("V4 CHECKPOINT VALIDATION")
print("=" * 80)

print("Checkpoint files:", len(files))

for f in files:
    print(
        os.path.basename(f),
        "|",
        f"{os.path.getsize(f)/(1024**2):.2f} MB"
    )


# ============================================================
# VALIDATION STATS
# ============================================================

total = 0
empty = 0
bad_json = 0
missing_doc_id = 0
missing_title = 0
duplicate_doc_ids = 0
duplicate_titles = 0

doc_ids = set()
titles = set()

telugu_counts = []
text_lengths = []

chunk_stats = {}

samples = defaultdict(list)


# ============================================================
# READ EVERY CHECKPOINT
# ============================================================

for file_index, path in enumerate(files):

    filename = os.path.basename(path)

    chunk_total = 0
    chunk_telugu = []
    chunk_lengths = []

    print()
    print("Checking:", filename)

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                row = json.loads(line)

            except Exception:
                bad_json += 1
                continue

            total += 1
            chunk_total += 1

            doc_id = row.get("doc_id")
            title = row.get("title")
            text = row.get("text", "")

            # ------------------------------------------------
            # Missing fields
            # ------------------------------------------------

            if not doc_id:
                missing_doc_id += 1

            if not title:
                missing_title += 1

            # ------------------------------------------------
            # Empty text
            # ------------------------------------------------

            if not text.strip():
                empty += 1

            # ------------------------------------------------
            # Duplicate doc ID
            # ------------------------------------------------

            if doc_id in doc_ids:
                duplicate_doc_ids += 1

            else:
                doc_ids.add(doc_id)

            # ------------------------------------------------
            # Duplicate titles
            # ------------------------------------------------

            if title in titles:
                duplicate_titles += 1

            else:
                titles.add(title)

            # ------------------------------------------------
            # Length statistics
            # ------------------------------------------------

            length = len(
                re.sub(
                    r"\s+",
                    "",
                    text
                )
            )

            telugu = sum(
                1
                for c in text
                if "\u0C00" <= c <= "\u0C7F"
            )

            text_lengths.append(length)
            telugu_counts.append(telugu)

            chunk_lengths.append(length)
            chunk_telugu.append(telugu)

            # ------------------------------------------------
            # Samples from beginning/middle/end
            # ------------------------------------------------

            if len(samples[file_index]) < 5:

                samples[file_index].append({
                    "title": title,
                    "text": text[:500]
                })

    chunk_stats[filename] = {
        "records": chunk_total,
        "avg_chars": (
            sum(chunk_lengths) / len(chunk_lengths)
            if chunk_lengths else 0
        ),
        "avg_telugu_chars": (
            sum(chunk_telugu) / len(chunk_telugu)
            if chunk_telugu else 0
        )
    }

    print(
        "Records:",
        f"{chunk_total:,}"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print()
print("=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(
    "Total records:",
    f"{total:,}"
)

print(
    "Bad JSON:",
    f"{bad_json:,}"
)

print(
    "Empty text:",
    f"{empty:,}"
)

print(
    "Missing doc_id:",
    f"{missing_doc_id:,}"
)

print(
    "Missing title:",
    f"{missing_title:,}"
)

print(
    "Duplicate doc_id:",
    f"{duplicate_doc_ids:,}"
)

print(
    "Duplicate titles:",
    f"{duplicate_titles:,}"
)

print()
print(
    "Unique doc_ids:",
    f"{len(doc_ids):,}"
)

print(
    "Unique titles:",
    f"{len(titles):,}"
)


# ============================================================
# LENGTH STATISTICS
# ============================================================

if text_lengths:

    sorted_lengths = sorted(text_lengths)

    n = len(sorted_lengths)

    median = sorted_lengths[n // 2]

    print()
    print("=" * 80)
    print("TEXT LENGTH STATISTICS")
    print("=" * 80)

    print(
        "Minimum chars:",
        f"{min(sorted_lengths):,}"
    )

    print(
        "Median chars:",
        f"{median:,}"
    )

    print(
        "Average chars:",
        f"{sum(sorted_lengths)/n:,.1f}"
    )

    print(
        "Maximum chars:",
        f"{max(sorted_lengths):,}"
    )


# ============================================================
# TELUGU STATISTICS
# ============================================================

if telugu_counts:

    sorted_telugu = sorted(
        telugu_counts
    )

    n = len(sorted_telugu)

    print()
    print("=" * 80)
    print("TELUGU CONTENT STATISTICS")
    print("=" * 80)

    print(
        "Minimum Telugu chars:",
        f"{min(sorted_telugu):,}"
    )

    print(
        "Median Telugu chars:",
        f"{sorted_telugu[n//2]:,}"
    )

    print(
        "Average Telugu chars:",
        f"{sum(sorted_telugu)/n:,.1f}"
    )

    print(
        "Maximum Telugu chars:",
        f"{max(sorted_telugu):,}"
    )


# ============================================================
# PER-CHUNK SUMMARY
# ============================================================

print()
print("=" * 80)
print("PER-CHECKPOINT SUMMARY")
print("=" * 80)

for filename, info in chunk_stats.items():

    print(
        f"{filename} | "
        f"records={info['records']:,} | "
        f"avg_chars={info['avg_chars']:,.0f} | "
        f"avg_telugu={info['avg_telugu_chars']:,.0f}"
    )


# ============================================================
# SAMPLE DOCUMENTS
# ============================================================

print()
print("=" * 80)
print("SAMPLE DOCUMENTS")
print("=" * 80)

for file_index in sorted(samples):

    filename = os.path.basename(
        files[file_index]
    )

    print()
    print("-" * 80)
    print(filename)
    print("-" * 80)

    for sample in samples[file_index]:

        print()
        print("TITLE:", sample["title"])
        print("TEXT:", sample["text"])


# ============================================================
# FINAL VERDICT
# ============================================================

print()
print("=" * 80)
print("VALIDATION VERDICT")
print("=" * 80)

if (
    bad_json == 0
    and empty == 0
    and missing_doc_id == 0
    and missing_title == 0
    and duplicate_doc_ids == 0
):

    print("PASS: Checkpoints are structurally clean.")

else:

    print("REVIEW REQUIRED: Problems detected.")

print("=" * 80)

V4 CHECKPOINT VALIDATION
Checkpoint files: 42
wikisource_v4_chunk_00000.jsonl | 29.51 MB
wikisource_v4_chunk_00001.jsonl | 22.38 MB
wikisource_v4_chunk_00002.jsonl | 29.15 MB
wikisource_v4_chunk_00003.jsonl | 1.38 MB
wikisource_v4_chunk_00004.jsonl | 1.53 MB
wikisource_v4_chunk_00005.jsonl | 0.09 MB
wikisource_v4_chunk_00006.jsonl | 0.16 MB
wikisource_v4_chunk_00007.jsonl | 0.11 MB
wikisource_v4_chunk_00008.jsonl | 0.09 MB
wikisource_v4_chunk_00009.jsonl | 0.38 MB
wikisource_v4_chunk_00010.jsonl | 1.86 MB
wikisource_v4_chunk_00011.jsonl | 1.39 MB
wikisource_v4_chunk_00012.jsonl | 0.14 MB
wikisource_v4_chunk_00013.jsonl | 0.07 MB
wikisource_v4_chunk_00014.jsonl | 0.14 MB
wikisource_v4_chunk_00015.jsonl | 0.32 MB
wikisource_v4_chunk_00016.jsonl | 0.03 MB
wikisource_v4_chunk_00017.jsonl | 0.04 MB
wikisource_v4_chunk_00018.jsonl | 0.01 MB
wikisource_v4_chunk_00019.jsonl | 0.01 MB
wikisource_v4_chunk_00020.jsonl | 0.11 MB
wikisource_v4_chunk_00021.jsonl | 0.02 MB
wikisource_v4_chunk_00022.j

In [ ]:
import os
import json
import glob
import hashlib

V4_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_checkpoints"
)

FIXED_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_fixed"
)

os.makedirs(FIXED_DIR, exist_ok=True)

files = sorted(
    glob.glob(
        os.path.join(
            V4_DIR,
            "wikisource_v4_chunk_*.jsonl"
        )
    )
)

total = 0
unique_ids = set()
duplicate_ids = 0

for path in files:

    filename = os.path.basename(path)

    output_path = os.path.join(
        FIXED_DIR,
        filename
    )

    chunk_count = 0

    with open(path, "r", encoding="utf-8") as inp, \
         open(output_path, "w", encoding="utf-8") as out:

        for line in inp:

            if not line.strip():
                continue

            row = json.loads(line)

            title = row.get("title", "")
            text = row.get("text", "")
            source = row.get(
                "source",
                "telugu_wikisource"
            )

            # Deterministic document ID
            key = (
                source
                + "\n"
                + title
                + "\n"
                + text
            )

            new_doc_id = hashlib.sha256(
                key.encode("utf-8")
            ).hexdigest()

            if new_doc_id in unique_ids:
                duplicate_ids += 1

            unique_ids.add(new_doc_id)

            row["doc_id"] = new_doc_id

            out.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

            total += 1
            chunk_count += 1

    print(
        f"{filename} | "
        f"records: {chunk_count:,}"
    )

print()
print("=" * 80)
print("DOC ID REPAIR COMPLETE")
print("=" * 80)

print("Total records:", f"{total:,}")
print("Unique IDs:", f"{len(unique_ids):,}")
print("Duplicate IDs:", f"{duplicate_ids:,}")
print("Output:", FIXED_DIR)

wikisource_v4_chunk_00000.jsonl | records: 4,656
wikisource_v4_chunk_00001.jsonl | records: 2,429
wikisource_v4_chunk_00002.jsonl | records: 994
wikisource_v4_chunk_00003.jsonl | records: 171
wikisource_v4_chunk_00004.jsonl | records: 31
wikisource_v4_chunk_00005.jsonl | records: 43
wikisource_v4_chunk_00006.jsonl | records: 20
wikisource_v4_chunk_00007.jsonl | records: 65
wikisource_v4_chunk_00008.jsonl | records: 5
wikisource_v4_chunk_00009.jsonl | records: 8
wikisource_v4_chunk_00010.jsonl | records: 58
wikisource_v4_chunk_00011.jsonl | records: 112
wikisource_v4_chunk_00012.jsonl | records: 18
wikisource_v4_chunk_00013.jsonl | records: 11
wikisource_v4_chunk_00014.jsonl | records: 8
wikisource_v4_chunk_00015.jsonl | records: 12
wikisource_v4_chunk_00016.jsonl | records: 8
wikisource_v4_chunk_00017.jsonl | records: 6
wikisource_v4_chunk_00018.jsonl | records: 3
wikisource_v4_chunk_00019.jsonl | records: 5
wikisource_v4_chunk_00020.jsonl | records: 5
wikisource_v4_chunk_00021.jsonl |

In [ ]:
import os
import json
import glob
import re
from collections import Counter

# ============================================================
# READ FROM FIXED DIRECTORY
# ============================================================

V4_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_fixed"
)

files = sorted(
    glob.glob(
        os.path.join(
            V4_DIR,
            "wikisource_v4_chunk_*.jsonl"
        )
    )
)

print("=" * 80)
print("VALIDATING FIXED V4 CORPUS")
print("=" * 80)

print("Checkpoint files:", len(files))

# ============================================================
# GLOBAL STATS
# ============================================================

total = 0
bad_json = 0
empty_text = 0
missing_doc_id = 0
missing_title = 0
duplicate_doc_ids = 0
duplicate_texts = 0

doc_ids = set()
texts = set()

text_lengths = []
telugu_counts = []

chunk_stats = {}

# ============================================================
# PROCESS EACH CHECKPOINT
# ============================================================

for path in files:

    filename = os.path.basename(path)

    chunk_records = 0
    chunk_lengths = []

    print()
    print("Checking:", filename)

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                row = json.loads(line)
            except Exception:
                bad_json += 1
                continue

            total += 1
            chunk_records += 1

            doc_id = row.get("doc_id")
            title = row.get("title")
            text = row.get("text", "")

            # ------------------------------------------------
            # Required fields
            # ------------------------------------------------

            if not doc_id:
                missing_doc_id += 1

            if not title:
                missing_title += 1

            # ------------------------------------------------
            # Duplicate IDs
            # ------------------------------------------------

            if doc_id in doc_ids:
                duplicate_doc_ids += 1
            else:
                doc_ids.add(doc_id)

            # ------------------------------------------------
            # Empty text
            # ------------------------------------------------

            if not text.strip():
                empty_text += 1

            # ------------------------------------------------
            # Duplicate text
            # ------------------------------------------------

            text_key = text.strip()

            if text_key in texts:
                duplicate_texts += 1
            else:
                texts.add(text_key)

            # ------------------------------------------------
            # Length
            # ------------------------------------------------

            clean_text = re.sub(
                r"\s+",
                "",
                text
            )

            length = len(clean_text)

            telugu = sum(
                1
                for c in text
                if "\u0C00" <= c <= "\u0C7F"
            )

            text_lengths.append(length)
            telugu_counts.append(telugu)
            chunk_lengths.append(length)

    chunk_stats[filename] = {
        "records": chunk_records,
        "avg_chars": (
            sum(chunk_lengths) / len(chunk_lengths)
            if chunk_lengths else 0
        )
    }

    print(
        "Records:",
        f"{chunk_records:,}"
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print()
print("=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(
    "Total records:",
    f"{total:,}"
)

print(
    "Bad JSON:",
    f"{bad_json:,}"
)

print(
    "Empty text:",
    f"{empty_text:,}"
)

print(
    "Missing doc_id:",
    f"{missing_doc_id:,}"
)

print(
    "Missing title:",
    f"{missing_title:,}"
)

print(
    "Duplicate doc_id:",
    f"{duplicate_doc_ids:,}"
)

print(
    "Duplicate text:",
    f"{duplicate_texts:,}"
)

print(
    "Unique doc_ids:",
    f"{len(doc_ids):,}"
)

print(
    "Unique texts:",
    f"{len(texts):,}"
)

# ============================================================
# LENGTH STATISTICS
# ============================================================

if text_lengths:

    sorted_lengths = sorted(text_lengths)
    n = len(sorted_lengths)

    print()
    print("=" * 80)
    print("TEXT LENGTH")
    print("=" * 80)

    print(
        "Minimum:",
        f"{min(sorted_lengths):,}"
    )

    print(
        "Median:",
        f"{sorted_lengths[n // 2]:,}"
    )

    print(
        "Average:",
        f"{sum(sorted_lengths) / n:,.1f}"
    )

    print(
        "Maximum:",
        f"{max(sorted_lengths):,}"
    )

# ============================================================
# TELUGU STATISTICS
# ============================================================

if telugu_counts:

    sorted_telugu = sorted(telugu_counts)
    n = len(sorted_telugu)

    print()
    print("=" * 80)
    print("TELUGU CONTENT")
    print("=" * 80)

    print(
        "Minimum Telugu chars:",
        f"{min(sorted_telugu):,}"
    )

    print(
        "Median Telugu chars:",
        f"{sorted_telugu[n // 2]:,}"
    )

    print(
        "Average Telugu chars:",
        f"{sum(sorted_telugu) / n:,.1f}"
    )

    print(
        "Maximum Telugu chars:",
        f"{max(sorted_telugu):,}"
    )

# ============================================================
# PER-CHUNK SUMMARY
# ============================================================

print()
print("=" * 80)
print("PER-CHUNK SUMMARY")
print("=" * 80)

for filename, info in chunk_stats.items():

    print(
        f"{filename} | "
        f"records={info['records']:,} | "
        f"avg_chars={info['avg_chars']:,.0f}"
    )

# ============================================================
# VERDICT
# ============================================================

print()
print("=" * 80)
print("VERDICT")
print("=" * 80)

if (
    bad_json == 0
    and empty_text == 0
    and missing_doc_id == 0
    and missing_title == 0
    and duplicate_doc_ids == 0
):

    print("PASS: Fixed V4 corpus is structurally valid.")

else:

    print("REVIEW REQUIRED: Problems detected.")

print("=" * 80)

VALIDATING FIXED V4 CORPUS
Checkpoint files: 42

Checking: wikisource_v4_chunk_00000.jsonl
Records: 4,656

Checking: wikisource_v4_chunk_00001.jsonl
Records: 2,429

Checking: wikisource_v4_chunk_00002.jsonl
Records: 994

Checking: wikisource_v4_chunk_00003.jsonl
Records: 171

Checking: wikisource_v4_chunk_00004.jsonl
Records: 31

Checking: wikisource_v4_chunk_00005.jsonl
Records: 43

Checking: wikisource_v4_chunk_00006.jsonl
Records: 20

Checking: wikisource_v4_chunk_00007.jsonl
Records: 65

Checking: wikisource_v4_chunk_00008.jsonl
Records: 5

Checking: wikisource_v4_chunk_00009.jsonl
Records: 8

Checking: wikisource_v4_chunk_00010.jsonl
Records: 58

Checking: wikisource_v4_chunk_00011.jsonl
Records: 112

Checking: wikisource_v4_chunk_00012.jsonl
Records: 18

Checking: wikisource_v4_chunk_00013.jsonl
Records: 11

Checking: wikisource_v4_chunk_00014.jsonl
Records: 8

Checking: wikisource_v4_chunk_00015.jsonl
Records: 12

Checking: wikisource_v4_chunk_00016.jsonl
Records: 8

Checking: w

In [ ]:
import os
import json
import glob

FIXED_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_fixed"
)

MERGED_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/final"
)

os.makedirs(MERGED_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    MERGED_DIR,
    "telugu_wikisource_v4_merged.jsonl"
)

files = sorted(
    glob.glob(
        os.path.join(
            FIXED_DIR,
            "wikisource_v4_chunk_*.jsonl"
        )
    )
)

total = 0

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as out:

    for path in files:

        chunk_count = 0

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as inp:

            for line in inp:

                if not line.strip():
                    continue

                # Validate JSON while merging
                row = json.loads(line)

                out.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

                total += 1
                chunk_count += 1

        print(
            f"{os.path.basename(path)} | "
            f"merged: {chunk_count:,}"
        )

print()
print("=" * 80)
print("MERGE COMPLETE")
print("=" * 80)

print(
    "Total records:",
    f"{total:,}"
)

print(
    "Output:",
    OUTPUT_FILE
)

print(
    "Size:",
    f"{os.path.getsize(OUTPUT_FILE)/(1024**2):.2f} MB"
)

wikisource_v4_chunk_00000.jsonl | merged: 4,656
wikisource_v4_chunk_00001.jsonl | merged: 2,429
wikisource_v4_chunk_00002.jsonl | merged: 994
wikisource_v4_chunk_00003.jsonl | merged: 171
wikisource_v4_chunk_00004.jsonl | merged: 31
wikisource_v4_chunk_00005.jsonl | merged: 43
wikisource_v4_chunk_00006.jsonl | merged: 20
wikisource_v4_chunk_00007.jsonl | merged: 65
wikisource_v4_chunk_00008.jsonl | merged: 5
wikisource_v4_chunk_00009.jsonl | merged: 8
wikisource_v4_chunk_00010.jsonl | merged: 58
wikisource_v4_chunk_00011.jsonl | merged: 112
wikisource_v4_chunk_00012.jsonl | merged: 18
wikisource_v4_chunk_00013.jsonl | merged: 11
wikisource_v4_chunk_00014.jsonl | merged: 8
wikisource_v4_chunk_00015.jsonl | merged: 12
wikisource_v4_chunk_00016.jsonl | merged: 8
wikisource_v4_chunk_00017.jsonl | merged: 6
wikisource_v4_chunk_00018.jsonl | merged: 3
wikisource_v4_chunk_00019.jsonl | merged: 5
wikisource_v4_chunk_00020.jsonl | merged: 5
wikisource_v4_chunk_00021.jsonl | merged: 6
wikisource

In [ ]:
import os
import json
import glob
import re
import random

V4_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/wikisource/v4_fixed"
)

files = sorted(
    glob.glob(
        os.path.join(
            V4_DIR,
            "wikisource_v4_chunk_*.jsonl"
        )
    )
)

records = []

for path in files:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

print("=" * 90)
print("FINAL TELUGU WIKISOURCE AUDIT")
print("=" * 90)

print("Checkpoint files :", len(files))
print("Total records    :", len(records))

# ------------------------------------------------------------
# BASIC VALIDATION
# ------------------------------------------------------------

doc_ids = [r.get("doc_id") for r in records]
titles = [r.get("title") for r in records]
texts = [r.get("text", "") for r in records]

print()
print("Unique doc_ids   :", len(set(doc_ids)))
print("Unique titles    :", len(set(titles)))
print("Unique texts     :", len(set(texts)))

print()
print("Missing doc_id   :", sum(not x for x in doc_ids))
print("Missing title    :", sum(not x for x in titles))
print("Empty text       :", sum(not x.strip() for x in texts))

# ------------------------------------------------------------
# TELUGU COUNTS
# ------------------------------------------------------------

def telugu_count(text):
    return sum(
        1 for c in text
        if "\u0C00" <= c <= "\u0C7F"
    )

telugu_counts = [
    telugu_count(t)
    for t in texts
]

lengths = [
    len(re.sub(r"\s+", "", t))
    for t in texts
]

print()
print("=" * 90)
print("CONTENT STATISTICS")
print("=" * 90)

print("Minimum characters       :", min(lengths))
print("Median characters        :", sorted(lengths)[len(lengths)//2])
print("Average characters       :", round(sum(lengths)/len(lengths), 2))
print("Maximum characters       :", max(lengths))

print()
print("Minimum Telugu chars     :", min(telugu_counts))
print("Median Telugu chars      :", sorted(telugu_counts)[len(telugu_counts)//2])
print("Average Telugu chars     :", round(sum(telugu_counts)/len(telugu_counts), 2))
print("Maximum Telugu chars     :", max(telugu_counts))

# ------------------------------------------------------------
# MEDIAWIKI REMNANTS
# ------------------------------------------------------------

patterns = {
    "templates": r"\{\{.*?\}\}",
    "wikilinks": r"\[\[.*?\]\]",
    "html": r"<[^>]+>",
    "category": r"(Category|వర్గం)\s*:",
}

print()
print("=" * 90)
print("MEDIAWIKI REMNANT CHECK")
print("=" * 90)

for name, pattern in patterns.items():

    count = sum(
        bool(re.search(pattern, t, flags=re.S | re.I))
        for t in texts
    )

    print(f"{name:15} :", count)

# ------------------------------------------------------------
# DUPLICATES
# ------------------------------------------------------------

from collections import Counter

text_counter = Counter(texts)

duplicate_groups = [
    (text, count)
    for text, count in text_counter.items()
    if count > 1
]

print()
print("=" * 90)
print("EXACT DUPLICATE CHECK")
print("=" * 90)

print("Duplicate groups :", len(duplicate_groups))
print(
    "Duplicate records:",
    sum(count - 1 for _, count in duplicate_groups)
)

# ------------------------------------------------------------
# SHORTEST DOCUMENTS
# ------------------------------------------------------------

indexed = list(enumerate(records))

shortest = sorted(
    indexed,
    key=lambda x: len(
        re.sub(r"\s+", "", x[1].get("text", ""))
    )
)[:5]

print()
print("=" * 90)
print("SHORTEST 5 DOCUMENTS")
print("=" * 90)

for i, row in shortest:

    text = row["text"]

    print()
    print("TITLE :", row["title"])
    print("ID    :", row["doc_id"])
    print("TELUGU:", telugu_count(text))
    print("TEXT  :", text[:500])

# ------------------------------------------------------------
# LONGEST DOCUMENTS
# ------------------------------------------------------------

longest = sorted(
    indexed,
    key=lambda x: len(
        re.sub(r"\s+", "", x[1].get("text", ""))
    ),
    reverse=True
)[:5]

print()
print("=" * 90)
print("LONGEST 5 DOCUMENTS")
print("=" * 90)

for i, row in longest:

    text = row["text"]

    print()
    print("TITLE :", row["title"])
    print("ID    :", row["doc_id"])
    print("CHARS :", len(re.sub(r"\s+", "", text)))
    print("TELUGU:", telugu_count(text))
    print("TEXT  :", text[:700])

# ------------------------------------------------------------
# RANDOM REAL DOCUMENTS
# ------------------------------------------------------------

random.seed(42)

sample_indices = random.sample(
    range(len(records)),
    min(10, len(records))
)

print()
print("=" * 90)
print("10 RANDOM REAL DOCUMENTS")
print("=" * 90)

for n, i in enumerate(sample_indices, 1):

    row = records[i]
    text = row["text"]

    print()
    print(f"--- SAMPLE {n} ---")
    print("TITLE :", row["title"])
    print("ID    :", row["doc_id"])
    print("CHARS :", len(re.sub(r"\s+", "", text)))
    print("TELUGU:", telugu_count(text))
    print("TEXT  :")
    print(text[:1000])

# ------------------------------------------------------------
# FINAL VERDICT
# ------------------------------------------------------------

print()
print("=" * 90)
print("FINAL AUDIT VERDICT")
print("=" * 90)

bad = []

if len(records) != 8909:
    bad.append("record count")

if len(set(doc_ids)) != 8909:
    bad.append("duplicate doc_ids")

if any(not x for x in doc_ids):
    bad.append("missing doc_ids")

if any(not x.strip() for x in texts):
    bad.append("empty texts")

if bad:
    print("REVIEW REQUIRED")
    print("Problems:", bad)
else:
    print("PASS")
    print("8,909 processed Wikisource records are structurally valid.")
    print("The 9 exact duplicate texts are retained intentionally.")
    print("They will be removed during the final cross-source deduplication.")

FINAL TELUGU WIKISOURCE AUDIT
Checkpoint files : 42
Total records    : 8909

Unique doc_ids   : 8909
Unique titles    : 8909
Unique texts     : 8900

Missing doc_id   : 0
Missing title    : 0
Empty text       : 0

CONTENT STATISTICS
Minimum characters       : 100
Median characters        : 1590
Average characters       : 3432.56
Maximum characters       : 388476

Minimum Telugu chars     : 50
Median Telugu chars      : 1490
Average Telugu chars     : 3112.02
Maximum Telugu chars     : 357467

MEDIAWIKI REMNANT CHECK
templates       : 0
wikilinks       : 17
html            : 0
category        : 10

EXACT DUPLICATE CHECK
Duplicate groups : 8
Duplicate records: 9

SHORTEST 5 DOCUMENTS

TITLE : ఉండమ్మా బొట్టు పెడతా!
ID    : be62c805e3c1511bb0312f25a6d6c82a2c23c0d689197028e18014fd0dc7ebe2
TELUGU: 90
TEXT  : ఉండమ్మా బొట్టు పెడతా (1968) సినిమా పాటలు.

# రావమ్మ మహలక్ష్మి రావమ్మా
# చాలులే నిదరపో జాబిలి కూన
# పాతాళ గంగమ్మా రారారా

TITLE : రాజు పేద
ID    : 61a2c2a9f820229821f7b32a776a7209b0a93b22

In [ ]:
import os

TEXTBOOK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/textbooks"
)

pdfs = []

for root, dirs, files in os.walk(TEXTBOOK_DIR):
    for file in files:
        if file.lower().endswith(".pdf"):
            path = os.path.join(root, file)
            size_mb = os.path.getsize(path) / (1024 ** 2)
            pdfs.append((path, size_mb))

print("=" * 80)
print("TELUGU TEXTBOOK COLLECTION")
print("=" * 80)

total_size = 0

for path, size_mb in sorted(pdfs):
    print(f"{size_mb:9.2f} MB | {path}")
    total_size += size_mb

print()
print("PDF files :", len(pdfs))
print("Total size:", f"{total_size / 1024:.2f} GB")

TELUGU TEXTBOOK COLLECTION
     9.30 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 10 telugu fl 2025-26.pdf
    36.46 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 106104128.pdf
   106.59 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 106105175.pdf
   101.08 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 106106179.pdf
    32.09 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 109104115.pdf
    80.63 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 10_telugu_textbook_2024-25.pdf
     6.59 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 10thtelugu om 2025-26.pdf
    88.24 MB | /content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of 110101132.pdf
   136.86 MB | /content/drive/MyDrive/LMA_Individual_Project/

In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!which pdftotext
!pdftotext -v

/usr/bin/pdftotext
pdftotext version 22.02.0
Copyright 2005-2022 The Poppler Developers - http://poppler.freedesktop.org
Copyright 1996-2011 Glyph & Cog, LLC


In [ ]:
import subprocess
import os

TEXTBOOK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/textbooks"
)

test_files = [
    "Copy of 1_Telugu_Textbook.pdf",
    "Copy of 9th Tel FL 2025-26.pdf"
]

for filename in test_files:

    path = os.path.join(TEXTBOOK_DIR, filename)

    print("=" * 80)
    print(filename)
    print("=" * 80)

    result = subprocess.run(
        [
            "pdftotext",
            "-f", "1",
            "-l", "3",
            "-layout",
            path,
            "-"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    print("RETURN CODE:", result.returncode)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr[:2000])

    print("\nTEXT PREVIEW:")
    print(result.stdout[:3000])

Copy of 1_Telugu_Textbook.pdf
RETURN CODE: 0

TEXT PREVIEW:
 Ôî\T>·<û\ jáTq <ûXø+T Ôî\T¹>qT
   Ôî\î T>·T e\¢uTó +& Ôî\î T>=¿£+&
jîT\¢ q|Ú\T ¿=\Te HîsTÁ >·yû u²kÍ&
     <ûXuø ²ó w\+<T Ôî\î T>·T ýÉdà
                                                                                                                      uó²sÁÔá sC²«+>·+
                                                                                                                     çbÍ<¸$T¿£ u²<ó«Ôá\T
                                                                           $uó²>·eTT - IVm, n~ó¿s£ DÁ + 51m
                                                                            ¿ì+~$ uó²sÁÔá<ûXø båsÁT\ u²<ó«Ôá\T :

                                                                           n     uó²sÁÔá sC²«+>±¿ì <TÆýÕÉ +&*. C²rjáT |Ô¿±, C²rjáT ^Ô, sC²«+>· XøjÖá \qT, sC²«+>·
                                                                                 d+d\qT >sÁ$+#*.
           

In [ ]:
!pip -q install paddleocr paddlepaddle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978

In [ ]:
!pip -q install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 27.4 MB/s eta 0:00:00


In [ ]:
import fitz

print("PyMuPDF:", fitz.version)

PyMuPDF: ('1.28.2', '1.28.2', None)


In [ ]:
import os
import fitz  # PyMuPDF

TEXTBOOK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/textbooks"
)

TEST_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/ocr_test"
)

os.makedirs(TEST_DIR, exist_ok=True)

test_files = [
    "Copy of 1_Telugu_Textbook.pdf",
    "Copy of 9th Tel FL 2025-26.pdf"
]

for filename in test_files:

    pdf_path = os.path.join(TEXTBOOK_DIR, filename)

    # Safe directory name
    book_name = os.path.splitext(filename)[0]
    book_name = book_name.replace(" ", "_")

    output_dir = os.path.join(
        TEST_DIR,
        book_name
    )

    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(pdf_path)

    pages_to_render = min(5, len(doc))

    print("=" * 80)
    print(filename)
    print("Total PDF pages:", len(doc))
    print("Rendering:", pages_to_render)

    for page_num in range(pages_to_render):

        page = doc[page_num]

        # ~200 DPI
        pix = page.get_pixmap(
            matrix=fitz.Matrix(2.5, 2.5),
            alpha=False
        )

        output_path = os.path.join(
            output_dir,
            f"page_{page_num + 1:03d}.png"
        )

        pix.save(output_path)

        print(
            f"Saved page {page_num + 1}: "
            f"{output_path}"
        )

    doc.close()

print()
print("=" * 80)
print("RENDER TEST COMPLETE")
print("=" * 80)

Copy of 1_Telugu_Textbook.pdf
Total PDF pages: 106
Rendering: 5
Saved page 1: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_001.png
Saved page 2: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_002.png
Saved page 3: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_003.png
Saved page 4: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_004.png
Saved page 5: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_005.png
Copy of 9th Tel FL 2025-26.pdf
Total PDF pages: 187
Rendering: 5
Saved page 1: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_9th_Tel_FL_2025-26/page_001.png
Saved page 2: /content/drive/M

In [ ]:
import glob
import os

images = glob.glob(
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/ocr_test/**/*.png",
    recursive=True
)

print("Rendered images:", len(images))

for x in images:
    print(
        f"{os.path.getsize(x)/(1024**2):.2f} MB | {x}"
    )

Rendered images: 10
6.41 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_001.png
2.31 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_002.png
0.45 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_003.png
0.26 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_004.png
0.12 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_1_Telugu_Textbook/page_005.png
4.55 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_9th_Tel_FL_2025-26/page_001.png
2.21 MB | /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/Copy_of_9th_Tel_FL_2025-26/page_002.png
0.10 MB | /content/drive/M

In [ ]:
import os

os.environ["FLAGS_enable_pir_api"] = "0"

print("PIR disabled:", os.environ["FLAGS_enable_pir_api"])

PIR disabled: 0


In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="te",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("Telugu OCR initialized successfully.")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('te_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/te_PP-OCRv5_mobile_rec`.


Telugu OCR initialized successfully.


In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-tel -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-tel.
(Reading database ... 118362 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-tel_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-tel (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-tel (1:4.00~git30-7274cfa-1.1) ...


In [ ]:
!tesseract --list-langs

List of available languages (3):
eng
osd
tel


In [ ]:
!pip -q install pytesseract

In [ ]:
import pytesseract
from PIL import Image

img = Image.open(
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/ocr_test/"
    "Copy_of_1_Telugu_Textbook/page_001.png"
)

text = pytesseract.image_to_string(
    img,
    lang="tel"
)

print(text[:5000])

  

తెలుగదేల యన్న దేశంబు తెలుగేను
తెలుగు వల్లభుండ తెలుగొకండ
యెల్ల నృపులు కొలువ నెరుగవే బాసాడి
దేశభాషలందు తెలుగు లెస్స

 

రాష్ట్ర విద్యా పరిశోధన శిక్షణా సంస్థ
ఆంధ్రప్రదేశ్‌



In [ ]:
import pytesseract
from PIL import Image
import glob
import os

IMAGE_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/ocr_test"
)

OCR_OUTPUT_DIR = os.path.join(
    IMAGE_DIR,
    "tesseract_output"
)

os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

images = sorted(
    glob.glob(
        os.path.join(IMAGE_DIR, "**", "*.png"),
        recursive=True
    )
)

# Don't accidentally OCR previously generated output images
images = [
    x for x in images
    if "tesseract_output" not in x
]

print("Images to OCR:", len(images))

for i, image_path in enumerate(images, 1):

    img = Image.open(image_path)

    text = pytesseract.image_to_string(
        img,
        lang="tel"
    )

    relative = os.path.relpath(
        image_path,
        IMAGE_DIR
    )

    output_name = (
        relative
        .replace(os.sep, "_")
        .replace(".png", ".txt")
    )

    output_path = os.path.join(
        OCR_OUTPUT_DIR,
        output_name
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(text)

    print(
        f"{i:02d}/{len(images)} | "
        f"chars={len(text):,} | "
        f"{relative}"
    )

print()
print("=" * 80)
print("10-PAGE OCR TEST COMPLETE")
print("=" * 80)

Images to OCR: 10
01/10 | chars=174 | Copy_of_1_Telugu_Textbook/page_001.png
02/10 | chars=1,833 | Copy_of_1_Telugu_Textbook/page_002.png
03/10 | chars=1,353 | Copy_of_1_Telugu_Textbook/page_003.png
04/10 | chars=793 | Copy_of_1_Telugu_Textbook/page_004.png
05/10 | chars=22 | Copy_of_1_Telugu_Textbook/page_005.png
06/10 | chars=1,168 | Copy_of_9th_Tel_FL_2025-26/page_001.png
07/10 | chars=2,979 | Copy_of_9th_Tel_FL_2025-26/page_002.png
08/10 | chars=251 | Copy_of_9th_Tel_FL_2025-26/page_003.png
09/10 | chars=825 | Copy_of_9th_Tel_FL_2025-26/page_004.png
10/10 | chars=2,944 | Copy_of_9th_Tel_FL_2025-26/page_005.png

10-PAGE OCR TEST COMPLETE


In [ ]:
import glob
import os

TELUGU_START = "\u0C00"
TELUGU_END = "\u0C7F"

txt_files = sorted(
    glob.glob(
        os.path.join(
            OCR_OUTPUT_DIR,
            "*.txt"
        )
    )
)

for path in txt_files:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        text = f.read()

    chars = [
        c for c in text
        if not c.isspace()
    ]

    telugu = sum(
        1 for c in chars
        if TELUGU_START <= c <= TELUGU_END
    )

    ratio = (
        telugu / len(chars)
        if chars else 0
    )

    print(
        f"{os.path.basename(path):55} "
        f"chars={len(chars):6,} "
        f"Telugu={ratio:.2%}"
    )

Copy_of_1_Telugu_Textbook_page_001.txt                  chars=   144 Telugu=99.31%
Copy_of_1_Telugu_Textbook_page_002.txt                  chars= 1,597 Telugu=95.37%
Copy_of_1_Telugu_Textbook_page_003.txt                  chars= 1,175 Telugu=88.43%
Copy_of_1_Telugu_Textbook_page_004.txt                  chars=   622 Telugu=24.60%
Copy_of_1_Telugu_Textbook_page_005.txt                  chars=     0 Telugu=0.00%
Copy_of_9th_Tel_FL_2025-26_page_001.txt                 chars=   938 Telugu=94.24%
Copy_of_9th_Tel_FL_2025-26_page_002.txt                 chars= 2,516 Telugu=86.37%
Copy_of_9th_Tel_FL_2025-26_page_003.txt                 chars=   208 Telugu=96.63%
Copy_of_9th_Tel_FL_2025-26_page_004.txt                 chars=   664 Telugu=22.59%
Copy_of_9th_Tel_FL_2025-26_page_005.txt                 chars= 2,622 Telugu=95.96%


In [ ]:
import os

for filename in [
    "Copy_of_1_Telugu_Textbook_page_004.txt",
    "Copy_of_1_Telugu_Textbook_page_005.txt",
    "Copy_of_9th_Tel_FL_2025-26_page_004.txt",
]:

    path = os.path.join(
        OCR_OUTPUT_DIR,
        filename
    )

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(text[:4000] if text else "[EMPTY OCR]")


Copy_of_1_Telugu_Textbook_page_004.txt
 

 

 

 

అ 6ఆ౦?ల౫౦ల601%06012 91260390. సం22726 శ

క7ా/ ౨4ల/బ/64 2020
గల 7207707 - 2020, 2022, 2023, 26024

ఉ1 11206 6030077661

౦ 021 0113 0011026010 92)/ 06 190106౪066,
510166 10 2 16016721 నయ, 01 [€205ముగం6, 10 ఖు?
10 009” ఖు? 96205 ₹గరం00 116 101 అయం101
0 రై 0% 16 0011396 101 06 0[యల౫గ56
010౪012196 10 ఖ్వు/ శం 00016108 01 00౪7౪ 0016
జు 0200 700100 119 08013066 ఖర 00302
9100114 00161601 20106108 015 0016101 0642
యం౦566 01 6 5003600601 8000256

౧6 00%” ౩౭1 10164 01 1113 0001 15 016
(01మమ1558100ల 0 501001 0002600. ముజుజా26
రమ 0126031.

 

(

10153 0001 [23 0600 001466 0% 70 65.)4. 55 34జ0[100
11606 2806 220 65.34. 019 గ ౮౪4

 

సమగ్ర శిక్షా, పాఠశాల విద్యాశాఖ,
ఆంధ్రప్రదేశ్‌ ప్రభుత్వం వారిచే ఉచిత పంపిణీ

 

0466 1 0612
264.9. ౮౦71 ల193001౬ 91635
తుంుజమా౭6
తర02 0126031

 

 

 

 


Copy_of_1_Telugu_Textbook_page_005.txt
 

   

 

 

   

 


Copy_of_9th_Tel_FL_2025-26_page_004.txt
 

అ ఆ౦?౪౧౫00%6౬01 1061209202. 0410120201

బా 4ల/స/64 20/౮
గల 

In [ ]:
import os
import glob
import hashlib
import fitz
import pandas as pd

TEXTBOOK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/textbooks"
)

MANIFEST_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks"
)

os.makedirs(MANIFEST_DIR, exist_ok=True)

pdfs = sorted(
    glob.glob(
        os.path.join(TEXTBOOK_DIR, "*.pdf")
    )
)

# Keywords for rough classification.
# This is only an initial classification; we will inspect uncertain files later.
TELUGU_KEYWORDS = [
    "telugu", "tel", "onamalu", "kollayi",
    "kashimajili", "amsumathi", "tenali",
    "తెనాలి", "తెలుగు", "subhaleka"
]

SUBJECT_KEYWORDS = [
    "physics", "chemistry", "biology",
    "history", "political", "economics",
    "business", "commerce", "accountancy",
    "community", "home science", "mass_comm",
    "geography", "psychology", "medical"
]

RELIGIOUS_KEYWORDS = [
    "geetha", "satcharitra", "punya"
]

rows = []

# Cache hashes so exact duplicate PDFs can be identified
hash_seen = {}

for i, path in enumerate(pdfs, 1):

    filename = os.path.basename(path)
    lower = filename.lower()

    size_bytes = os.path.getsize(path)
    size_mb = size_bytes / (1024 ** 2)

    # Exact file hash
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            sha256.update(chunk)

    file_hash = sha256.hexdigest()

    duplicate_of = ""

    if file_hash in hash_seen:
        duplicate_of = hash_seen[file_hash]
    else:
        hash_seen[file_hash] = filename

    # Rough category
    if any(k in lower for k in RELIGIOUS_KEYWORDS):
        category = "religious/literary"

    elif any(k in lower for k in SUBJECT_KEYWORDS):
        category = "subject_textbook"

    elif any(k in lower for k in TELUGU_KEYWORDS):
        category = "telugu_language"

    else:
        category = "unknown"

    # PDF metadata
    try:
        doc = fitz.open(path)

        page_count = len(doc)

        metadata = doc.metadata or {}
        pdf_title = metadata.get("title", "") or ""

        doc.close()

    except Exception as e:
        page_count = -1
        pdf_title = ""
        print("PDF error:", filename, e)

    rows.append({
        "filename": filename,
        "size_mb": round(size_mb, 2),
        "pages": page_count,
        "category_initial": category,
        "exact_file_hash": file_hash,
        "duplicate_of": duplicate_of,
        "pdf_title": pdf_title,
        "decision": ""
    })

    print(
        f"{i:02d}/{len(pdfs)} | "
        f"{page_count:5} pages | "
        f"{size_mb:8.2f} MB | "
        f"{category:20} | "
        f"{filename}"
    )

manifest = pd.DataFrame(rows)

manifest_path = os.path.join(
    MANIFEST_DIR,
    "textbook_collection_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

print()
print("=" * 80)
print("MANIFEST CREATED")
print("=" * 80)
print("PDFs:", len(manifest))
print("Total size:", f"{manifest['size_mb'].sum()/1024:.2f} GB")
print("Exact duplicate files:",
      (manifest["duplicate_of"] != "").sum())
print("Manifest:", manifest_path)

01/73 |   182 pages |     9.30 MB | telugu_language      | Copy of 10 telugu fl 2025-26.pdf
02/73 |   526 pages |    36.46 MB | unknown              | Copy of 106104128.pdf
03/73 |  1012 pages |   106.59 MB | unknown              | Copy of 106105175.pdf
04/73 |   824 pages |   101.08 MB | unknown              | Copy of 106106179.pdf
05/73 |   662 pages |    32.09 MB | unknown              | Copy of 109104115.pdf
06/73 |   210 pages |    80.63 MB | telugu_language      | Copy of 10_telugu_textbook_2024-25.pdf
07/73 |   118 pages |     6.59 MB | telugu_language      | Copy of 10thtelugu om 2025-26.pdf
08/73 |   323 pages |    88.24 MB | unknown              | Copy of 110101132.pdf
09/73 |   106 pages |   136.86 MB | telugu_language      | Copy of 1_Telugu_Textbook.pdf
10/73 |   132 pages |    15.27 MB | subject_textbook     | Copy of 2015.391960.MEDICAL-BOOK.pdf
11/73 |   118 pages |   122.66 MB | telugu_language      | Copy of 2_Telugu_Textbook.pdf
12/73 |   300 pages |     2.10 MB | te

In [ ]:
print("\nCATEGORY SUMMARY")
print("=" * 80)

print(
    manifest["category_initial"]
    .value_counts()
    .to_string()
)

print("\nEXACT DUPLICATES")
print("=" * 80)

duplicates = manifest[
    manifest["duplicate_of"] != ""
][[
    "filename",
    "duplicate_of",
    "size_mb"
]]

print(
    duplicates.to_string(index=False)
    if len(duplicates)
    else "No exact duplicate files found."
)


CATEGORY SUMMARY
category_initial
telugu_language       33
subject_textbook      28
unknown                9
religious/literary     3

EXACT DUPLICATES
                             filename                              duplicate_of  size_mb
Copy of Class-2 Telugu FL 2025-26.pdf Copy of Class-2 Telugu FL 2025-26 (1).pdf    53.92
Copy of Class-3 Telugu FL 2025-26.pdf Copy of Class-3 Telugu FL 2025-26 (1).pdf    92.07


In [ ]:
unknown = manifest[
    manifest["category_initial"] == "unknown"
][["filename", "pages", "size_mb"]]

print("=" * 100)
print("UNKNOWN FILES")
print("=" * 100)

print(
    unknown.to_string(index=False)
)

UNKNOWN FILES
                                 filename  pages  size_mb
                    Copy of 106104128.pdf    526    36.46
                    Copy of 106105175.pdf   1012   106.59
                    Copy of 106106179.pdf    824   101.08
                    Copy of 109104115.pdf    662    32.09
                    Copy of 110101132.pdf    323    88.24
Copy of Vyapara Ganakam Book Complete.pdf    525     2.32
             Copy of pg39561-images-3.pdf    168     1.26
             Copy of pg43220-images-3.pdf    254     1.82
             Copy of pg50178-images-3.pdf    327     2.34


In [ ]:
import os
import fitz

UNKNOWN_TEST_DIR = os.path.join(
    MANIFEST_DIR,
    "classification_test"
)

os.makedirs(UNKNOWN_TEST_DIR, exist_ok=True)

unknown_files = unknown["filename"].tolist()

for filename in unknown_files:

    pdf_path = os.path.join(
        TEXTBOOK_DIR,
        filename
    )

    safe_name = (
        os.path.splitext(filename)[0]
        .replace(" ", "_")
        .replace("/", "_")
    )

    output_dir = os.path.join(
        UNKNOWN_TEST_DIR,
        safe_name
    )

    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(pdf_path)

    # First and middle page where possible
    page_indices = [0]

    if len(doc) > 1:
        page_indices.append(len(doc) // 2)

    for page_num in page_indices:

        page = doc[page_num]

        pix = page.get_pixmap(
            matrix=fitz.Matrix(2.0, 2.0),
            alpha=False
        )

        output_path = os.path.join(
            output_dir,
            f"page_{page_num + 1:04d}.png"
        )

        pix.save(output_path)

    doc.close()

print("Classification test pages created.")

Classification test pages created.


In [ ]:
import pytesseract
from PIL import Image
import glob
import os

UNKNOWN_TEST_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/classification_test"
)

OCR_DIR = os.path.join(
    UNKNOWN_TEST_DIR,
    "ocr"
)

os.makedirs(OCR_DIR, exist_ok=True)

images = sorted(
    glob.glob(
        os.path.join(
            UNKNOWN_TEST_DIR,
            "**",
            "*.png"
        ),
        recursive=True
    )
)

print("Images found:", len(images))

for i, image_path in enumerate(images, 1):

    img = Image.open(image_path)

    text = pytesseract.image_to_string(
        img,
        lang="tel"
    )

    filename = os.path.basename(
        os.path.dirname(image_path)
    )

    page_name = os.path.splitext(
        os.path.basename(image_path)
    )[0]

    output_dir = os.path.join(
        OCR_DIR,
        filename
    )

    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        page_name + ".txt"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(text)

    print(
        f"{i:02d}/{len(images)} | "
        f"chars={len(text):,} | "
        f"{filename}/{page_name}"
    )

print()
print("=" * 80)
print("UNKNOWN-FILE OCR TEST COMPLETE")
print("=" * 80)

Images found: 18
01/18 | chars=145 | Copy_of_106104128/page_0001
02/18 | chars=300 | Copy_of_106104128/page_0264
03/18 | chars=556 | Copy_of_106105175/page_0001
04/18 | chars=1,090 | Copy_of_106105175/page_0507
05/18 | chars=603 | Copy_of_106106179/page_0001
06/18 | chars=1,671 | Copy_of_106106179/page_0413
07/18 | chars=131 | Copy_of_109104115/page_0001
08/18 | chars=1,793 | Copy_of_109104115/page_0332
09/18 | chars=270 | Copy_of_110101132/page_0001
10/18 | chars=753 | Copy_of_110101132/page_0162
11/18 | chars=272 | Copy_of_Vyapara_Ganakam_Book_Complete/page_0001
12/18 | chars=1,239 | Copy_of_Vyapara_Ganakam_Book_Complete/page_0263
13/18 | chars=40 | Copy_of_pg39561-images-3/page_0001
14/18 | chars=1,105 | Copy_of_pg39561-images-3/page_0085
15/18 | chars=34 | Copy_of_pg43220-images-3/page_0001
16/18 | chars=1,128 | Copy_of_pg43220-images-3/page_0128
17/18 | chars=64 | Copy_of_pg50178-images-3/page_0001
18/18 | chars=1,127 | Copy_of_pg50178-images-3/page_0164

UNKNOWN-FILE OCR TEST COM

In [ ]:
import glob
import os

txt_files = sorted(
    glob.glob(
        os.path.join(
            OCR_DIR,
            "**",
            "*.txt"
        ),
        recursive=True
    )
)

for path in txt_files:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        text = f.read()

    print("\n" + "=" * 100)
    print(
        os.path.relpath(
            path,
            OCR_DIR
        )
    )
    print("=" * 100)

    print(
        text[:1500]
        if text.strip()
        else "[EMPTY OCR]"
    )


Copy_of_106104128/page_0001.txt
 

న ల... వా త్త ఫెళ

.
న
కక: (అ 90 అక్ర అ; 1కి ౧6,6౯: [౯ ఈ

 

౧౦ 520/2646/ 1120064216410021

6౦౧౧౮16౧ 5016౧06 ౧06 ౬001౧66100
11] గ200061 ]

 


Copy_of_106104128/page_0264.txt
 

 

 

 

 

 

 

 

 

 

 

గీ[86006౧ ఆర్జుమెంటన్‌ వాదనలు

10816 లాజిక్‌ తర్మం

1000 లూప్‌ ఉచ్చు

0/08౯20016 ప్రోగామ్‌ కర్వక్రమము / కార్వక్రమం
1౧061 ఇన్నుట్‌ ప్రవేశం

౦409641 అవుట్సుట్‌ బహీర్లథం _/ ఉతృత్తి
౪2౧12016 వేరియబుల్‌ అస్టిరమ'ఇన _చరరాశి
502160%060€ స్టేట్మెంట్‌ పంక్తి _ ప్రకటన

 

263


Copy_of_106105175/page_0001.txt
య రా

   
 

౧౦1 ౧౩2౧౦౧౩ ౧౧2౧1౧0) 025
౧0% 523001౧230 య౧౧౩ఈ0౦౧26|౧3౫/2)/
౧0% గ264511.020ఈ023

6౧౦౧౧౧౮౪౪6 5016౧66 ౩౧౮6 ౬౧91౧66౧౧౦, 117 క౧౩/౧39006

 

 

 

 

౧6౧5106166 0౫
(వ 1౧౨11౮6
౧౯౦౦44౦ ౧4144 1621541014 ౪112 /16114014070711 157707౬౯0౯ ౬1611౬౬116 7౬61006౪
1440147144 /0714184 ౮౬౬౯ 7౯41164014 84010//420 645565 ౪/5౬౯42౬ ౧౬51౦౯౧14౬
[1110004 60కంంగం 0౬౯6౧౯౯౦౦౬.86౬౯౯౦౧౪/౦014౯&
1౬ 8424711 [ఎ
5కటుటాం 8 454/1 00౬86౯0౯ 9161౫5౯౧116
1/ఆ1166 0)
- 1౧౨11౮8
0౧14/

In [ ]:
import os
import fitz
import pandas as pd

SUBJECT_TEST_DIR = os.path.join(
    MANIFEST_DIR,
    "subject_classification_test"
)

os.makedirs(SUBJECT_TEST_DIR, exist_ok=True)

subject_files = manifest[
    manifest["category_initial"] == "subject_textbook"
]["filename"].tolist()

print("Subject PDFs:", len(subject_files))

for filename in subject_files:

    pdf_path = os.path.join(
        TEXTBOOK_DIR,
        filename
    )

    safe_name = (
        os.path.splitext(filename)[0]
        .replace(" ", "_")
        .replace("/", "_")
    )

    output_dir = os.path.join(
        SUBJECT_TEST_DIR,
        safe_name
    )

    os.makedirs(output_dir, exist_ok=True)

    try:
        doc = fitz.open(pdf_path)
        n = len(doc)

        # Beginning, middle, and later page
        page_indices = sorted(set([
            0,
            n // 2,
            n - 1
        ]))

        for page_num in page_indices:

            page = doc[page_num]

            pix = page.get_pixmap(
                matrix=fitz.Matrix(2.0, 2.0),
                alpha=False
            )

            output_path = os.path.join(
                output_dir,
                f"page_{page_num + 1:04d}.png"
            )

            pix.save(output_path)

        doc.close()

        print(
            f"OK | {filename} | "
            f"{n} pages | sampled {len(page_indices)}"
        )

    except Exception as e:

        print(
            f"ERROR | {filename} | {e}"
        )

print()
print("=" * 80)
print("SUBJECT CLASSIFICATION PAGES CREATED")
print("=" * 80)

Subject PDFs: 28
OK | Copy of 2015.391960.MEDICAL-BOOK.pdf | 132 pages | sampled 3
OK | Copy of 312 Physics Vol - I TM.pdf | 408 pages | sampled 3
OK | Copy of 312 Physics Vol - II TM.pdf | 420 pages | sampled 3
OK | Copy of 312_PHYSICS_TM.pdf | 451 pages | sampled 3
OK | Copy of 313_Inter_Chemistry_Practical_Manual_TM_2024.pdf | 169 pages | sampled 3
OK | Copy of 313_Inter_Chemistry_Vol_1_TM.pdf | 450 pages | sampled 3
OK | Copy of 313_Inter_Chemistry_Vol_2_TM.pdf | 481 pages | sampled 3
OK | Copy of 313_Inter_Chemistry_Vol_3_TM.pdf | 189 pages | sampled 3
OK | Copy of 314_BIOLOGY_TM.pdf | 534 pages | sampled 3
OK | Copy of 315_HISTORY_TM.pdf | 213 pages | sampled 3
OK | Copy of 315_Intermediate_History_TM.pdf | 278 pages | sampled 3
OK | Copy of 317_POLITICAL SCIENCE_TM.pdf | 253 pages | sampled 3
OK | Copy of 318_Inter_Economics_TM.pdf | 389 pages | sampled 3
OK | Copy of 319_BUSINESS STUDIES_TM.pdf | 305 pages | sampled 3
OK | Copy of 319_Commerce_Business_Studies_TM.pdf | 512 page

In [ ]:
import pytesseract
from PIL import Image
import glob
import os

SUBJECT_OCR_DIR = os.path.join(
    SUBJECT_TEST_DIR,
    "ocr"
)

os.makedirs(SUBJECT_OCR_DIR, exist_ok=True)

images = sorted(
    glob.glob(
        os.path.join(
            SUBJECT_TEST_DIR,
            "**",
            "*.png"
        ),
        recursive=True
    )
)

print("Images found:", len(images))

for i, image_path in enumerate(images, 1):

    img = Image.open(image_path)

    text = pytesseract.image_to_string(
        img,
        lang="tel"
    )

    book_name = os.path.basename(
        os.path.dirname(image_path)
    )

    page_name = os.path.splitext(
        os.path.basename(image_path)
    )[0]

    output_dir = os.path.join(
        SUBJECT_OCR_DIR,
        book_name
    )

    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        page_name + ".txt"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(text)

    print(
        f"{i:03d}/{len(images)} | "
        f"chars={len(text):,} | "
        f"{book_name}/{page_name}"
    )

print()
print("=" * 80)
print("SUBJECT OCR SAMPLING COMPLETE")
print("=" * 80)

Images found: 84
001/84 | chars=16 | Copy_of_2015.391960.MEDICAL-BOOK/page_0001
002/84 | chars=1,376 | Copy_of_2015.391960.MEDICAL-BOOK/page_0067
003/84 | chars=717 | Copy_of_2015.391960.MEDICAL-BOOK/page_0132
004/84 | chars=377 | Copy_of_312_PHYSICS_TM/page_0001
005/84 | chars=2,109 | Copy_of_312_PHYSICS_TM/page_0226
006/84 | chars=664 | Copy_of_312_PHYSICS_TM/page_0451
007/84 | chars=748 | Copy_of_312_Physics_Vol_-_II_TM/page_0001
008/84 | chars=1,035 | Copy_of_312_Physics_Vol_-_II_TM/page_0211
009/84 | chars=356 | Copy_of_312_Physics_Vol_-_II_TM/page_0420
010/84 | chars=747 | Copy_of_312_Physics_Vol_-_I_TM/page_0001
011/84 | chars=1,911 | Copy_of_312_Physics_Vol_-_I_TM/page_0205
012/84 | chars=296 | Copy_of_312_Physics_Vol_-_I_TM/page_0408
013/84 | chars=59 | Copy_of_313_Inter_Chemistry_Practical_Manual_TM_2024/page_0001
014/84 | chars=1,405 | Copy_of_313_Inter_Chemistry_Practical_Manual_TM_2024/page_0085
015/84 | chars=1,940 | Copy_of_313_Inter_Chemistry_Practical_Manual_TM_2024/pa

In [ ]:
import glob
import os

def telugu_ratio(text):

    non_space = [
        c for c in text
        if not c.isspace()
    ]

    if not non_space:
        return 0.0

    telugu = sum(
        1
        for c in non_space
        if "\u0C00" <= c <= "\u0C7F"
    )

    return telugu / len(non_space)


txt_files = sorted(
    glob.glob(
        os.path.join(
            SUBJECT_OCR_DIR,
            "**",
            "*.txt"
        ),
        recursive=True
    )
)

for path in txt_files:

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        text = f.read()

    print(
        f"{os.path.relpath(path, SUBJECT_OCR_DIR):70} "
        f"chars={len(text):6,} "
        f"Telugu={telugu_ratio(text):6.2%}"
    )

Copy_of_2015.391960.MEDICAL-BOOK/page_0001.txt                         chars=    16 Telugu=100.00%
Copy_of_2015.391960.MEDICAL-BOOK/page_0067.txt                         chars= 1,376 Telugu=88.72%
Copy_of_2015.391960.MEDICAL-BOOK/page_0132.txt                         chars=   717 Telugu=84.64%
Copy_of_312_PHYSICS_TM/page_0001.txt                                   chars=   377 Telugu=62.50%
Copy_of_312_PHYSICS_TM/page_0226.txt                                   chars= 2,109 Telugu=94.82%
Copy_of_312_PHYSICS_TM/page_0451.txt                                   chars=   664 Telugu=86.89%
Copy_of_312_Physics_Vol_-_II_TM/page_0001.txt                          chars=   748 Telugu=85.67%
Copy_of_312_Physics_Vol_-_II_TM/page_0211.txt                          chars= 1,035 Telugu=77.31%
Copy_of_312_Physics_Vol_-_II_TM/page_0420.txt                          chars=   356 Telugu=74.83%
Copy_of_312_Physics_Vol_-_I_TM/page_0001.txt                           chars=   747 Telugu=85.65%
Copy_of_312_Physics

In [ ]:
# Create the processing manifest for the textbook collection.

PROCESSING_MANIFEST_PATH = os.path.join(
    MANIFEST_DIR,
    "textbook_processing_manifest.csv"
)

processing = manifest.copy()

# Exact duplicate files are excluded from processing.
processing["process"] = "YES"
processing.loc[
    processing["duplicate_of"] != "",
    "process"
] = "NO_DUPLICATE"

# All currently classified categories are candidates.
processing["source_group"] = processing["category_initial"]

# Update the nine files we manually inspected.
unknown_decisions = {
    "Copy of 106104128.pdf": "telugu_subject",
    "Copy of 106105175.pdf": "telugu_subject",
    "Copy of 106106179.pdf": "telugu_subject",
    "Copy of 109104115.pdf": "telugu_subject",
    "Copy of 110101132.pdf": "telugu_subject",
    "Copy of Vyapara Ganakam Book Complete.pdf": "telugu_subject",
    "Copy of pg39561-images-3.pdf": "telugu_literature",
    "Copy of pg43220-images-3.pdf": "telugu_literature",
    "Copy of pg50178-images-3.pdf": "telugu_literature",
}

for filename, group in unknown_decisions.items():
    processing.loc[
        processing["filename"] == filename,
        "source_group"
    ] = group

processing.to_csv(
    PROCESSING_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("TEXTBOOK PROCESSING MANIFEST")
print("=" * 80)

print(
    processing[
        ["filename", "pages", "source_group", "process"]
    ].to_string(index=False)
)

print()
print("Total PDFs:", len(processing))
print(
    "Excluded exact duplicates:",
    (processing["process"] == "NO_DUPLICATE").sum()
)
print(
    "PDFs to process:",
    (processing["process"] == "YES").sum()
)
print()
print("Output:")
print(PROCESSING_MANIFEST_PATH)

TEXTBOOK PROCESSING MANIFEST
                                                             filename  pages       source_group      process
                                     Copy of 10 telugu fl 2025-26.pdf    182    telugu_language          YES
                                                Copy of 106104128.pdf    526     telugu_subject          YES
                                                Copy of 106105175.pdf   1012     telugu_subject          YES
                                                Copy of 106106179.pdf    824     telugu_subject          YES
                                                Copy of 109104115.pdf    662     telugu_subject          YES
                               Copy of 10_telugu_textbook_2024-25.pdf    210    telugu_language          YES
                                    Copy of 10thtelugu om 2025-26.pdf    118    telugu_language          YES
                                                Copy of 110101132.pdf    323     telugu_subject    

In [ ]:
# Normalize source groups before OCR

processing = pd.read_csv(
    PROCESSING_MANIFEST_PATH,
    encoding="utf-8-sig"
)

processing.loc[
    processing["source_group"] == "subject_textbook",
    "source_group"
] = "telugu_subject"

processing.loc[
    processing["source_group"] == "religious/literary",
    "source_group"
] = "telugu_literary"

processing.to_csv(
    PROCESSING_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("NORMALIZED SOURCE GROUPS")
print("=" * 80)

print(
    processing["source_group"]
    .value_counts()
    .to_string()
)

print()
print("PDFs to process:",
      (processing["process"] == "YES").sum())

NORMALIZED SOURCE GROUPS
source_group
telugu_subject       34
telugu_language      33
telugu_literary       3
telugu_literature     3

PDFs to process: 71


In [ ]:
# Finalize source-group naming

processing = pd.read_csv(
    PROCESSING_MANIFEST_PATH,
    encoding="utf-8-sig"
)

processing.loc[
    processing["source_group"] == "telugu_literary",
    "source_group"
] = "telugu_literature"

processing.to_csv(
    PROCESSING_MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("FINAL SOURCE GROUPS")
print("=" * 80)

print(
    processing["source_group"]
    .value_counts()
    .to_string()
)

print()
print("PDFs to process:",
      (processing["process"] == "YES").sum())

FINAL SOURCE GROUPS
source_group
telugu_subject       34
telugu_language      33
telugu_literature     6

PDFs to process: 71


In [ ]:
import os
import json
import re
import glob
import fitz
import pytesseract
import pandas as pd

# ============================================================
# PATHS
# ============================================================

TEXTBOOK_DIR = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/raw/manual/textbooks"
)

MANIFEST_PATH = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/"
    "textbook_processing_manifest.csv"
)

OCR_ROOT = (
    "/content/drive/MyDrive/LMA_Individual_Project/"
    "data/processed/manual/textbooks/"
    "ocr_checkpoints"
)

os.makedirs(OCR_ROOT, exist_ok=True)

# ============================================================
# LOAD MANIFEST
# ============================================================

processing = pd.read_csv(
    MANIFEST_PATH,
    encoding="utf-8-sig"
)

processing = processing[
    processing["process"] == "YES"
].copy()

processing = processing.sort_values(
    "filename"
).reset_index(drop=True)

print("=" * 80)
print("TEXTBOOK OCR PIPELINE")
print("=" * 80)
print("PDFs to process:", len(processing))
print("Output:", OCR_ROOT)

TEXTBOOK OCR PIPELINE
PDFs to process: 71
Output: /content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_checkpoints


In [ ]:
import time
from PIL import Image
import io

# ============================================================
# BASIC QUALITY SIGNALS
# ============================================================

def telugu_ratio(text):

    chars = [
        c for c in text
        if not c.isspace()
    ]

    if not chars:
        return 0.0

    telugu = sum(
        1
        for c in chars
        if "\u0C00" <= c <= "\u0C7F"
    )

    return telugu / len(chars)


def digit_symbol_ratio(text):

    chars = [
        c for c in text
        if not c.isspace()
    ]

    if not chars:
        return 1.0

    bad = sum(
        1
        for c in chars
        if c.isdigit()
        or (
            not c.isalpha()
            and not ("\u0C00" <= c <= "\u0C7F")
        )
    )

    return bad / len(chars)


# ============================================================
# PROCESS EACH PDF
# ============================================================

for pdf_index, row in processing.iterrows():

    filename = row["filename"]

    pdf_path = os.path.join(
        TEXTBOOK_DIR,
        filename
    )

    # One checkpoint file per PDF.
    # This makes resume behaviour very simple and safe.
    safe_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        os.path.splitext(filename)[0]
    )

    output_path = os.path.join(
        OCR_ROOT,
        safe_name + ".jsonl"
    )

    # --------------------------------------------------------
    # Find already completed pages
    # --------------------------------------------------------

    completed_pages = set()

    if os.path.exists(output_path):

        with open(
            output_path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                if not line.strip():
                    continue

                try:
                    record = json.loads(line)

                    completed_pages.add(
                        int(record["page"])
                    )

                except Exception:
                    # Ignore incomplete/corrupt final line.
                    continue

    # --------------------------------------------------------
    # Open PDF
    # --------------------------------------------------------

    try:
        doc = fitz.open(pdf_path)

    except Exception as e:

        print(
            f"\nERROR opening: {filename}"
        )
        print(e)
        continue

    total_pages = len(doc)

    print()
    print("=" * 80)
    print(
        f"[{pdf_index + 1}/{len(processing)}] "
        f"{filename}"
    )
    print(
        f"Pages: {total_pages} | "
        f"Already done: {len(completed_pages)}"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # Process pages
    # --------------------------------------------------------

    for page_index in range(total_pages):

        page_number = page_index + 1

        # Resume support
        if page_number in completed_pages:
            continue

        start = time.time()

        try:

            page = doc[page_index]

            # 2x rendering.
            # We deliberately don't use huge DPI because
            # the PDFs can be very large.
            pix = page.get_pixmap(
                matrix=fitz.Matrix(2.0, 2.0),
                alpha=False
            )

            # Convert pixmap → PIL image
            img = Image.open(
                io.BytesIO(
                    pix.tobytes("png")
                )
            )

            # Telugu OCR
            text = pytesseract.image_to_string(
                img,
                lang="tel"
            )

            text = text.strip()

            # ------------------------------------------------
            # Basic page signals
            # ------------------------------------------------

            chars = len(text)

            tel_ratio = telugu_ratio(text)

            bad_ratio = digit_symbol_ratio(text)

            # We are NOT aggressively filtering here.
            # These labels are diagnostic only.
            if chars == 0:
                status = "EMPTY"

            elif chars < 80:
                status = "VERY_SHORT"

            elif tel_ratio < 0.20:
                status = "LOW_TELUGU"

            else:
                status = "CANDIDATE"

            record = {
                "doc_id": None,
                "source": "manual_textbooks",
                "book": filename,
                "page": page_number,
                "text": text,
                "char_count": chars,
                "telugu_ratio": round(
                    tel_ratio,
                    6
                ),
                "digit_symbol_ratio": round(
                    bad_ratio,
                    6
                ),
                "status": status
            }

            # ------------------------------------------------
            # Write immediately
            # ------------------------------------------------

            with open(
                output_path,
                "a",
                encoding="utf-8"
            ) as out:

                out.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

            elapsed = time.time() - start

            print(
                f"page {page_number:4d}/{total_pages} | "
                f"chars={chars:6,} | "
                f"Telugu={tel_ratio:6.2%} | "
                f"{status:10} | "
                f"{elapsed:5.1f}s"
            )

        except Exception as e:

            print(
                f"ERROR | page {page_number} | {e}"
            )

    doc.close()

    print(
        f"COMPLETED: {filename}"
    )

print()
print("=" * 80)
print("OCR EXTRACTION FINISHED")
print("=" * 80)


[1/71] Copy of 10 telugu fl 2025-26.pdf
Pages: 182 | Already done: 182
COMPLETED: Copy of 10 telugu fl 2025-26.pdf

[2/71] Copy of 106104128.pdf
Pages: 526 | Already done: 526
COMPLETED: Copy of 106104128.pdf

[3/71] Copy of 106105175.pdf
Pages: 1012 | Already done: 909
page  910/1012 | chars= 1,487 | Telugu=56.52% | CANDIDATE  |  12.8s
page  911/1012 | chars= 1,293 | Telugu=77.37% | CANDIDATE  |   5.9s
page  912/1012 | chars= 1,715 | Telugu=75.59% | CANDIDATE  |  10.8s
page  913/1012 | chars=   802 | Telugu=68.72% | CANDIDATE  |   4.1s
page  914/1012 | chars=   732 | Telugu=74.76% | CANDIDATE  |   4.5s
page  915/1012 | chars= 1,127 | Telugu=64.20% | CANDIDATE  |   6.6s
page  916/1012 | chars= 2,052 | Telugu=66.40% | CANDIDATE  |  11.5s
page  917/1012 | chars= 1,128 | Telugu=71.61% | CANDIDATE  |   5.5s
page  918/1012 | chars= 1,516 | Telugu=51.70% | CANDIDATE  |   9.8s
page  919/1012 | chars= 2,022 | Telugu=74.74% | CANDIDATE  |  12.4s
page  920/1012 | chars= 1,433 | Telugu=76.48% | 


KeyboardInterrupt



In [ ]:
import os

BASE = "/content/drive/MyDrive/LMA_Individual_Project"

for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith((".jsonl", ".txt", ".csv")) and (
            "ocr" in root.lower()
            or "textbook" in root.lower()
        ):
            print(os.path.join(root, f))

/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of output (1).txt
/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks/Copy of output.txt
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/textbook_collection_manifest.csv
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/textbook_processing_manifest.csv
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/tesseract_output/Copy_of_1_Telugu_Textbook_page_001.txt
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/tesseract_output/Copy_of_1_Telugu_Textbook_page_002.txt
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/tesseract_output/Copy_of_1_Telugu_Textbook_page_003.txt
/content/drive/MyDrive/LMA_Individual_Project/data/processed/manual/textbooks/ocr_test/tesseract_output/Copy_of_1_Telugu_Textbook_page_004.txt
/content/drive/MyDri

In [ ]:
import os

pdf_dir = "/content/drive/MyDrive/LMA_Individual_Project/data/raw/manual/textbooks"

print("PDF files:")
for f in sorted(os.listdir(pdf_dir)):
    if f.lower().endswith(".pdf"):
        print(f)

PDF files:
Copy of 10 telugu fl 2025-26.pdf
Copy of 106104128.pdf
Copy of 106105175.pdf
Copy of 106106179.pdf
Copy of 109104115.pdf
Copy of 10_telugu_textbook_2024-25.pdf
Copy of 10thtelugu om 2025-26.pdf
Copy of 110101132.pdf
Copy of 1_Telugu_Textbook.pdf
Copy of 2015.391960.MEDICAL-BOOK.pdf
Copy of 2_Telugu_Textbook.pdf
Copy of 305_Telugu.pdf
Copy of 312 Physics Vol - I TM.pdf
Copy of 312 Physics Vol - II TM.pdf
Copy of 312_PHYSICS_TM.pdf
Copy of 313_Inter_Chemistry_Practical_Manual_TM_2024.pdf
Copy of 313_Inter_Chemistry_Vol_1_TM.pdf
Copy of 313_Inter_Chemistry_Vol_2_TM.pdf
Copy of 313_Inter_Chemistry_Vol_3_TM.pdf
Copy of 314_BIOLOGY_TM.pdf
Copy of 315_HISTORY_TM.pdf
Copy of 315_Intermediate_History_TM.pdf
Copy of 317_POLITICAL SCIENCE_TM.pdf
Copy of 318_Inter_Economics_TM.pdf
Copy of 319_BUSINESS STUDIES_TM.pdf
Copy of 319_Commerce_Business_Studies_TM.pdf
Copy of 320_ACCOUNTANCY_1 TM.pdf
Copy of 320_ACCOUNTANCY_2 TM.pdf
Copy of 320_ACCOUNTANCY_3 TM.pdf
Copy of 321_INTE_COMMUNITY_SC

In [ ]:
import os
import json

checkpoint_dir = (
    "/content/drive/MyDrive/"
    "LMA_Individual_Project/data/processed/manual/textbooks/"
    "ocr_checkpoints"
)

files = [
    "Copy_of_106104128.jsonl",
    "Copy_of_106105175.jsonl",
    "Copy_of_106106179.jsonl",
    "Copy_of_10_telugu_fl_2025-26.jsonl",
]

for filename in files:
    path = os.path.join(checkpoint_dir, filename)

    print("\n" + "=" * 80)
    print(filename)

    if not os.path.exists(path):
        print("NOT FOUND")
        continue

    count = 0
    pages = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
                try:
                    record = json.loads(line)
                    if "page" in record:
                        pages.append(record["page"])
                except:
                    pass

    print("JSONL records :", count)

    if pages:
        print("Minimum page :", min(pages))
        print("Maximum page :", max(pages))
        print("Unique pages :", len(set(pages)))

        expected = max(pages)

        missing = sorted(
            set(range(1, expected + 1))
            - set(pages)
        )

        print("Missing within range :", len(missing))

        if missing:
            print(
                "First missing pages:",
                missing[:20]
            )


Copy_of_106104128.jsonl
JSONL records : 526
Minimum page : 1
Maximum page : 526
Unique pages : 526
Missing within range : 0

Copy_of_106105175.jsonl
JSONL records : 1012
Minimum page : 1
Maximum page : 1012
Unique pages : 1012
Missing within range : 0

Copy_of_106106179.jsonl
JSONL records : 248
Minimum page : 1
Maximum page : 248
Unique pages : 248
Missing within range : 0

Copy_of_10_telugu_fl_2025-26.jsonl
JSONL records : 182
Minimum page : 1
Maximum page : 182
Unique pages : 182
Missing within range : 0
